# Análisis de ECG: ¿GPU o CPU en producción?

Este notebook mide, con el código real del pipeline, cuánto tarda un análisis en CPU y en GPU, comprueba que la GPU no cambia la calidad y, con esos tiempos, estima cuánto esperaría cada estudiante si unos 10 envían su ECG a la vez en clase.

Qué hace, en orden:

1. Comprueba el entorno: qué GPU ha asignado Colab, CUDA y la versión de PyTorch.
2. Prepara el código: `ecg-pipeline` en `main`, Open-ECG-Digitizer clonado y parcheado con `scripts/setup_digitizer.sh`, los pesos de ECGFounder y las dependencias con versiones fijadas.
3. Genera las imágenes de validación: 12 registros públicos de PTB-XL (fold 10, PhysioNet, CC BY 4.0) dibujados con ECG-Image-Kit, en versión limpia y distorsionada. Son los 12 del estudio de escalas de `docs/RENDIMIENTO.md`.
4. Mide en CPU y en GPU: tiempo por etapa, tiempo por estudio (el primero, en frío, y los siguientes) en los dos modos del digitalizador, y memoria pico.
5. Compara la calidad de CPU y GPU con la misma tolerancia del estudio de escalas.
6. Simula la clase en tres formas de servir el análisis y resume la decisión en una tabla.
7. Deja un CSV, un JSON y la tabla final listos para descargar.

Datos: todo se genera aquí dentro a partir de datos públicos. No subas a Colab imágenes de pacientes ni las de estudios de `api-EKG/media`.

Cómo ejecutarlo:

1. Menú Entorno de ejecución > Cambiar tipo de entorno > Acelerador de hardware: T4 GPU > Guardar.
2. Menú Entorno de ejecución > Ejecutar todas.
3. Deja la pestaña abierta hasta que termine. Al final se descarga `resultados_gpu_cpu.zip`.

Tiempo aproximado con la T4 gratuita: de 1 h 30 a 2 h 30 en total. Preparar lleva unos 10 a 15 minutos, las imágenes de 5 a 10, la GPU unos 15 y la mayor parte se va en la CPU: Colab da 2 vCPU y cada imagen tarda de 2 a 4 minutos. Con `MODO_RAPIDO = True` (2 registros) baja a unos 30 o 40 minutos, útil para una primera prueba.

Si Colab se desconecta a mitad pero conserva la máquina, vuelve a ejecutar todo: las mediciones que ya tienen su JSON no se repiten.

## 1. Entorno

La CPU es el procesador normal, el que tendría cualquier servidor. La GPU es una tarjeta gráfica que hace las multiplicaciones de las redes neuronales en paralelo, mucho más rápido. `nvidia-smi` es el programa que lista las GPU de la máquina y CUDA es la capa que permite a PyTorch usarlas.

Si la celda avisa de que no hay GPU, actívala en Entorno de ejecución > Cambiar tipo de entorno > T4 GPU y vuelve a empezar. Si no la activas, el notebook sigue y mide solo la CPU.

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path


def salida_de(cmd):
    """Salida de un comando, o cadena vacía si no existe o falla."""
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    except (OSError, subprocess.TimeoutExpired):
        return ""
    return r.stdout.strip() if r.returncode == 0 else ""


HAY_GPU = bool(salida_de(["nvidia-smi", "-L"]))
if HAY_GPU:
    print(salida_de(["nvidia-smi"]))
    GPU_NOMBRE = salida_de(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).splitlines()[0]
    DRIVER = salida_de(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]).splitlines()[0]
else:
    GPU_NOMBRE, DRIVER = None, None
    print("AVISO: esta máquina no tiene GPU.")
    print("Para activarla: Entorno de ejecución > Cambiar tipo de entorno > Acelerador de hardware: T4 GPU > Guardar.")
    print("Después vuelve a ejecutar el notebook desde el principio. Si no la activas, se mide solo la CPU.")

CPUS = len(os.sched_getaffinity(0))
with open("/proc/meminfo") as fh:
    RAM_GB = int(fh.readline().split()[1]) / 1024**2
with open("/proc/cpuinfo") as fh:
    MODELO_CPU = next((l.split(":", 1)[1].strip() for l in fh if l.startswith("model name")), "desconocido")
print(f"\nCPU: {CPUS} vCPU visibles, {MODELO_CPU}")
print(f"RAM: {RAM_GB:.1f} GB   Python de Colab: {platform.python_version()}")
if HAY_GPU:
    print(f"GPU: {GPU_NOMBRE}, driver {DRIVER}")

try:
    import torch

    print(f"PyTorch de Colab: {torch.__version__}, CUDA {torch.version.cuda}, GPU visible: {torch.cuda.is_available()}")
except ImportError:
    print("Este entorno de Colab no trae PyTorch.")
print("Las mediciones no usan el PyTorch de Colab sino un entorno aparte con versiones fijadas (sección 2),")
print("y allí se vuelve a comprobar CUDA.")

## Parámetros

Los valores por defecto reproducen el estudio de escalas; lo normal es no tocar nada.

- `MODO_RAPIDO`: con `True` se miden 2 registros (4 imágenes) en vez de 12, para una prueba corta. Las conclusiones valen con `False`.
- `HILOS_CPU`: cuántos hilos de CPU usa PyTorch. Se fija de forma explícita y es el mismo en todas las mediciones, también en las de GPU, porque parte del digitalizador corre en CPU aunque haya GPU. Llega a los procesos por `ECG_TORCH_THREADS`, `OMP_NUM_THREADS` y `MKL_NUM_THREADS`, y el perfil por etapas llama además a `torch.set_num_threads`. El valor es el de las vCPU visibles con el tope de 8 que aplica el pipeline por defecto (`ecg_pipeline/threads.py`). En Colab gratis son 2 vCPU, un solo núcleo físico.
- `SECUENCIA_ESTUDIOS`: los estudios seguidos con los que se mide el arranque en frío. La primera imagen se repite al final para separar el coste del arranque de lo que cambia de una imagen a otra.
- Simulación: `N_ESTUDIANTES`, `WORKERS_VM` y los máximos de instancias de Cloud Run. `ARRANQUE_CONTENEDOR_S` son los segundos que tarda la plataforma en levantar el contenedor antes de que arranque Python; aquí no se puede medir y por defecto es 0.

In [ ]:
# Código que se mide. main es lo que corre en producción.
RAMA_PIPELINE = "main"
# Open-ECG-Digitizer: el commit sobre el que se validaron los parches 0001 a 0004 y el estudio de escalas.
COMMIT_DIGITALIZADOR = "963387ff5abdfa3db91c15ac52d6cf1214345a6a"
# ECG-Image-Kit: el commit con el que se generaron las imágenes de la validación.
COMMIT_IMAGE_KIT = "27b90f56896c9fc78b05a83ca14844ea2637aa0b"

MODO_RAPIDO = False
HILOS_CPU = max(1, min(CPUS, 8))
SECUENCIA_ESTUDIOS = ["limpio/1172.png", "distorsionado/8215.png", "limpio/1172.png"]
PNG_MUESTRAS = 2  # imágenes en las que se mide aparte la imagen de depuración
REPETIR = False  # True: repite también las mediciones que ya tienen su JSON

# Simulación de la clase.
N_ESTUDIANTES = 10
WORKERS_VM = 2
MAX_INSTANCIAS_CPU = 10  # máximo de instancias del servicio de Cloud Run en CPU
MAX_INSTANCIAS_GPU = 10  # máximo de instancias con GPU: revisa la cuota de GPU del proyecto
ARRANQUE_CONTENEDOR_S = 0.0

# Los 12 registros del estudio de escalas (docs/RENDIMIENTO.md, sección 2): los cinco grupos
# diagnósticos y los distorsionados más difíciles de la auditoría.
REGISTROS_ESTUDIO = {
    "1172": "NORM", "1355": "NORM", "8645": "NORM",
    "7221": "AFIB", "8215": "AFIB", "17386": "AFIB",
    "17690": "CRBBB_CLBBB", "2433": "CRBBB_CLBBB",
    "6231": "IMI_AMI", "18506": "IMI_AMI",
    "19424": "SBRAD_STACH", "8507": "SBRAD_STACH",
}  # fmt: skip

# Los 24 registros de la validación. Se generan los 24 aunque se midan 12: ECG-Image-Kit fija
# la semilla una vez por lote y cada registro consume números aleatorios, así que generar solo
# los 12 cambiaría las distorsiones de las imágenes.
REGISTROS_VALIDACION = {
    "618": "CRBBB_CLBBB", "1172": "NORM", "1355": "NORM", "1487": "NORM", "2419": "SBRAD_STACH",
    "2433": "CRBBB_CLBBB", "3484": "SBRAD_STACH", "4951": "NORM", "5946": "NORM", "5992": "NORM",
    "6231": "IMI_AMI", "7221": "AFIB", "8215": "AFIB", "8507": "SBRAD_STACH", "8645": "NORM",
    "17386": "AFIB", "17690": "CRBBB_CLBBB", "17744": "IMI_AMI", "18421": "IMI_AMI",
    "18506": "IMI_AMI", "18548": "CRBBB_CLBBB", "18847": "NORM", "19424": "SBRAD_STACH", "20406": "AFIB",
}  # fmt: skip

REGISTROS = {k: REGISTROS_ESTUDIO[k] for k in ("1172", "8215")} if MODO_RAPIDO else dict(REGISTROS_ESTUDIO)
VARIANTES = ("limpio", "distorsionado")

CONTENT = Path("/content")
REPO = CONTENT / "ecg-pipeline"
DIGITALIZADOR = CONTENT / "Open-ECG-Digitizer"
KIT = CONTENT / "ecg-image-kit"
VENV_PY = CONTENT / "venv-ecg" / "bin" / "python"
VENV_KIT_PY = CONTENT / "venv-kit" / "bin" / "python"
MEDIR = CONTENT / "medir"
VAL = CONTENT / "val"
PTBXL = VAL / "ptbxl"
IMAGENES = VAL / "imagenes"
RESULTADOS = CONTENT / "resultados"
PESOS_1LEAD = REPO / "weights" / "1_lead_ECGFounder.pth"
# Los umbrales provisionales de fold 10, para que ECGFounder marque hallazgos como en api-EKG.
UMBRALES = REPO / "configs" / "thresholds" / "provisional" / "fold10_2026-09-11"
for carpeta in (MEDIR, VAL, RESULTADOS):
    carpeta.mkdir(parents=True, exist_ok=True)

os.environ["RAMA_PIPELINE"] = RAMA_PIPELINE
os.environ["COMMIT_DIGITALIZADOR"] = COMMIT_DIGITALIZADOR
os.environ["COMMIT_IMAGE_KIT"] = COMMIT_IMAGE_KIT
DISPOSITIVOS = ["cpu"] + (["cuda"] if HAY_GPU else [])
FALLOS = {}
print(f"Registros: {len(REGISTROS)} x 2 variantes = {2 * len(REGISTROS)} imágenes. Hilos de CPU: {HILOS_CPU}.")
print(f"Dispositivos a medir: {', '.join(DISPOSITIVOS)}")

## 2. Preparar el código y las dependencias

Tres repositorios y dos entornos de Python separados, creados con `uv`:

- `ecg-pipeline` en `main`.
- Open-ECG-Digitizer en el commit validado. Se clona con git-lfs, porque sus pesos llegan por LFS y sin él se bajan punteros de texto en vez de modelos. Después `scripts/setup_digitizer.sh` le aplica los parches 0001 a 0004; el 0004 añade el servidor persistente `src/serve.py`. `scripts/download_weights.sh` baja los pesos de ECGFounder, unos 740 MB.
- ECG-Image-Kit, solo para dibujar las imágenes.

Las versiones van fijadas. `requirements.txt` exige `numpy<2` y `torch<2.4`, así que el pipeline usa torch 2.3.1, cuya compilación de PyPI trae CUDA 12.1 y corre igual en CPU: se mide el mismo binario en los dos dispositivos. El resto son las versiones con las que se validó el digitalizador. ECG-Image-Kit necesita Python 3.10 y va en su propio entorno, sin tensorflow: su módulo de texto manuscrito no se usa.

Tarda unos 10 a 15 minutos, casi todo descargas.

In [ ]:
%%bash
set -euo pipefail
cd /content

echo "==> Herramientas"
command -v uv >/dev/null || pip install -q uv
if ! command -v git-lfs >/dev/null; then
  apt-get -qq update && apt-get -qq install -y git-lfs >/dev/null
fi
git lfs install --skip-repo >/dev/null

echo "==> ecg-pipeline ($RAMA_PIPELINE)"
[ -d ecg-pipeline ] || git clone -q --branch "$RAMA_PIPELINE" https://github.com/reeenatamc/ecg-pipeline.git
git -C ecg-pipeline log -1 --format='    %h %s'

echo "==> Open-ECG-Digitizer ($COMMIT_DIGITALIZADOR)"
if [ ! -f Open-ECG-Digitizer/src/digitize.py ]; then
  GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/Ahus-AIM/Open-ECG-Digitizer.git
  git -C Open-ECG-Digitizer checkout -q "$COMMIT_DIGITALIZADOR"
  git -C Open-ECG-Digitizer lfs pull
fi
# El checkout ya existe, así que el script no vuelve a clonar: solo aplica los parches, en orden.
OPEN_ECG_DIGITIZER_HOME=/content/Open-ECG-Digitizer bash ecg-pipeline/scripts/setup_digitizer.sh | tee /content/setup_digitizer.log
if grep -q "WARNING" /content/setup_digitizer.log; then
  echo "ERROR: algún parche no se aplicó limpio; revisa el log de arriba."
  exit 1
fi
[ -f Open-ECG-Digitizer/src/serve.py ] || { echo "ERROR: falta src/serve.py (parche 0004)."; exit 1; }
for w in Open-ECG-Digitizer/weights/*.pt; do
  tam=$(stat -c %s "$w")
  [ "$tam" -gt 1000000 ] || { echo "ERROR: $w es un puntero de LFS, no los pesos."; exit 1; }
  echo "    $(basename "$w"): $((tam / 1048576)) MB"
done

echo "==> Pesos de ECGFounder"
bash ecg-pipeline/scripts/download_weights.sh 2>&1 | grep -v '#' || true
ls -la ecg-pipeline/weights

echo "==> ECG-Image-Kit ($COMMIT_IMAGE_KIT)"
if [ ! -d ecg-image-kit ]; then
  git clone -q https://github.com/alphanumericslab/ecg-image-kit.git
  git -C ecg-image-kit checkout -q "$COMMIT_IMAGE_KIT"
fi
echo "Listo."

In [ ]:
%%bash
set -euo pipefail
cd /content

echo "==> Entorno del pipeline: Python 3.12, torch 2.3.1 (CUDA 12.1)"
[ -x venv-ecg/bin/python ] || uv venv -q --python 3.12 venv-ecg
uv pip install -q --python venv-ecg/bin/python \
  torch==2.3.1 torchvision==0.18.1 numpy==1.26.4 scipy==1.12.0 scikit-image==0.22.0 \
  scikit-learn==1.4.2 networkx==3.2.1 matplotlib==3.8.4 pillow==10.2.0 pyyaml==6.0.1 \
  tqdm==4.67.1 yacs==0.1.8 torch-tps==1.2.2 wfdb==4.1.2 pandas==2.2.2
# Sin dependencias: ya están arriba con sus versiones, y así nada mueve el par numpy/torch.
uv pip install -q --python venv-ecg/bin/python --no-deps -e ecg-pipeline

echo "==> Entorno de ECG-Image-Kit: Python 3.10, sin tensorflow"
[ -x venv-kit/bin/python ] || uv venv -q --python 3.10 venv-kit
uv pip install -q --python venv-kit/bin/python \
  numpy==1.26.4 scipy==1.13.1 pandas==2.2.2 matplotlib==3.8.4 pillow==10.3.0 \
  scikit-image==0.21.0 opencv-python==4.6.0.66 imgaug==0.4.0 imageio==2.27.0 \
  imutils==0.5.4 qrcode==7.4.2 wfdb==4.1.2 pyyaml==6.0.3 shapely==2.1.2
echo "Listo."

In [ ]:
comprobacion = subprocess.run(
    [
        str(VENV_PY),
        "-c",
        "import json, numpy, torch, ecg_pipeline; print(json.dumps({"
        "'torch': torch.__version__, 'cuda_torch': torch.version.cuda, "
        "'cuda_disponible': torch.cuda.is_available(), "
        "'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "
        "'numpy': numpy.__version__, 'ecg_pipeline': getattr(ecg_pipeline, '__version__', 'desconocida')}))",
    ],
    capture_output=True,
    text=True,
)
if comprobacion.returncode != 0:
    print(comprobacion.stderr[-3000:])
    raise RuntimeError("El entorno del pipeline no importa torch o ecg_pipeline; revisa la sección 2.")
ENTORNO_VENV = json.loads(comprobacion.stdout.strip().splitlines()[-1])
print(json.dumps(ENTORNO_VENV, indent=2))

if HAY_GPU and not ENTORNO_VENV["cuda_disponible"]:
    print("AVISO: hay GPU pero el torch del entorno no la ve. Se mide solo la CPU.")
    DISPOSITIVOS = ["cpu"]
    FALLOS["cuda"] = "torch del entorno sin CUDA"

ENTORNO = {
    "gpu": GPU_NOMBRE,
    "driver": DRIVER,
    "cpu_modelo": MODELO_CPU,
    "vcpu": CPUS,
    "ram_gb": round(RAM_GB, 1),
    "hilos_cpu": HILOS_CPU,
    "entorno_pipeline": ENTORNO_VENV,
    "commit_pipeline": salida_de(["git", "-C", str(REPO), "rev-parse", "HEAD"]),
    "commit_digitalizador": salida_de(["git", "-C", str(DIGITALIZADOR), "rev-parse", "HEAD"]),
    "commit_image_kit": salida_de(["git", "-C", str(KIT), "rev-parse", "HEAD"]),
}
print(f"ecg-pipeline {ENTORNO['commit_pipeline'][:7]}, digitalizador {ENTORNO['commit_digitalizador'][:7]}")

### Herramientas de medición

Cuatro scripts pequeños que se escriben en `/content/medir` y corren en el entorno del pipeline. Ninguno modifica el pipeline ni el digitalizador: los importan y cronometran.

- `generar_kit.py` lanza el generador de ECG-Image-Kit sin su módulo de texto manuscrito y con los archivos en orden alfabético, para que el resultado no dependa del sistema de archivos.
- `medir_etapas.py` es el perfil por etapas de `docs/RENDIMIENTO.md`: reproduce paso a paso lo que hace el digitalizador con cada imagen y cronometra cada parte. En GPU sincroniza antes y después de cada etapa; si no, el tiempo de una etapa se anotaría en la siguiente.
- `medir_estudios.py` procesa estudios seguidos igual que el worker de api-EKG: `pipeline.run` para digitalizar y después ECGFounder con el modelo cargado una vez.
- `calidad.py` compara cada CSV con la señal original de PTB-XL y la CPU con la GPU.

In [ ]:
%%writefile /content/medir/generar_kit.py
"""Ejecuta el generador por lotes de ECG-Image-Kit sin su módulo de texto manuscrito.

HandwrittenText/generate.py importa tensorflow y spacy al cargarse, pero los comandos de la
validación nunca piden texto manuscrito (-l/--link), así que esa función no se llama. Un
módulo vacío en su lugar evita instalar varios GB sin tocar el código del kit.

El kit fija la semilla una sola vez al principio del lote, así que la secuencia aleatoria que
recibe cada registro depende del orden en que los recorre. Ese orden sale de os.walk y de
os.listdir, que dependen del sistema de archivos; aquí se ordenan alfabéticamente para que
dos ejecuciones en Colab generen lo mismo.

    python generar_kit.py DIR_DEL_GENERADOR [argumentos de gen_ecg_images_from_data_batch.py]
"""

import os
import runpy
import sys
import types


def _sin_manuscrito(*args, **kwargs):
    raise RuntimeError("Texto manuscrito desactivado en este entorno.")


def main() -> None:
    gen_dir = os.path.abspath(sys.argv[1])

    modulo = types.ModuleType("HandwrittenText.generate")
    modulo.get_handwritten = _sin_manuscrito
    paquete = types.ModuleType("HandwrittenText")
    paquete.__path__ = []
    sys.modules["HandwrittenText"] = paquete
    sys.modules["HandwrittenText.generate"] = modulo

    walk_original = os.walk
    listdir_original = os.listdir

    def walk_ordenado(top, topdown=True, onerror=None, followlinks=False):
        for raiz, carpetas, archivos in walk_original(top, True, onerror, followlinks):
            carpetas.sort()
            archivos.sort()
            yield raiz, carpetas, archivos

    def listdir_ordenado(path="."):
        return sorted(listdir_original(path))

    os.walk = walk_ordenado
    os.listdir = listdir_ordenado

    os.chdir(gen_dir)
    sys.path.insert(0, gen_dir)
    script = os.path.join(gen_dir, "gen_ecg_images_from_data_batch.py")
    sys.argv = [script] + sys.argv[2:]
    runpy.run_path(script, run_name="__main__")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/medir/medir_etapas.py
"""Perfil por etapas de la digitalización y de ECGFounder, en un solo proceso, en CPU o en GPU.

Mismo método que docs/RENDIMIENTO.md, sección 1: reproduce src.digitize.process_one_file del
digitalizador (decodificar, semilla por imagen, InferenceWrapper, guardar) y cronometra
también lo que el digitalizador no cronometra: imports, inicio de CUDA, carga de pesos,
identificación de formato y cada escritura. Después pasa las compuertas de calidad de
ecg_pipeline y ECGFounder (camino rhythm) sobre cada CSV.

En GPU las operaciones son asíncronas: sin sincronizar, el tiempo de una etapa se anotaría en
la siguiente que espera un resultado. Por eso cada cronómetro sincroniza la GPU al empezar y
al terminar, también los del propio digitalizador.

La imagen de depuración no se escribe dentro del tiempo del estudio (la configuración llega
con DATA.save_mode = timeseries_only). Su coste se mide aparte, en las primeras
--png-muestras imágenes, en una carpeta separada.

    python medir_etapas.py --digitalizador DIR --config CFG --imagenes DIR --salida DIR \
        --dispositivo cpu|cuda --hilos N --json FILE \
        [--pesos CKPT] [--png-muestras K] [--interpretar-extra DIR_CSV]
"""

from __future__ import annotations

import argparse
import contextlib
import json
import math
import os
import resource
import sys
import time
from pathlib import Path
from typing import Any

T_INICIO = time.perf_counter()

SUFIJO_CSV = "_timeseries_canonical.csv"

# Nombres de las secciones que cronometra el propio InferenceWrapper, en español y en el
# orden en que se ejecutan.
ETAPAS_ES = {
    "Initial resampling": "Remuestreo inicial",
    "Segmentation": "Segmentación (U-Net)",
    "Perspective detection": "Detección de perspectiva",
    "Cropping": "Recorte",
    "Feature map resampling": "Remuestreo de mapas",
    "Pixel size search": "Búsqueda de tamaño de píxel",
    "Dewarping": "Corrección de deformación (desactivada)",
    "Signal extraction": "Extracción de señal",
}


def limpiar_json(valor: Any) -> Any:
    """NaN e infinitos a None, para que el JSON lo lea cualquier herramienta."""
    if isinstance(valor, float) and not math.isfinite(valor):
        return None
    if isinstance(valor, dict):
        return {str(k): limpiar_json(v) for k, v in valor.items()}
    if isinstance(valor, (list, tuple)):
        return [limpiar_json(v) for v in valor]
    return valor


def rss_pico_mb(quien: int) -> float:
    rss = resource.getrusage(quien).ru_maxrss
    return rss / (1 << 20) if sys.platform == "darwin" else rss / 1024


def resumen_observaciones(resultado: dict[str, Any]) -> dict[str, Any]:
    """Lo que ECGFounder dijo de un registro, sin el resto del resultado."""
    return {
        "topk": resultado.get("topk"),
        "marcadas": resultado.get("flagged"),
        "origen_umbral": resultado.get("threshold_source"),
        "derivaciones_ritmo": resultado.get("rhythm_leads"),
        "degradado": resultado.get("degraded"),
        "error": resultado.get("error"),
    }


def compuertas_registro(pipeline: Any, meta: dict[str, Any] | None, calidad: dict[str, Any]) -> dict[str, Any]:
    """Las compuertas de ecg_pipeline, igual que pipeline.run(skip_interpretation=True)."""
    _, plantilla_degradada, plantilla_compuertas = pipeline._layout_template_warnings(meta, calidad)
    compuertas = [pipeline.GATE_LAYOUT_UNKNOWN] if pipeline._layout_failed(meta) else []
    compuertas += list(calidad.get("gates", []))
    compuertas += plantilla_compuertas
    completas = list(calidad.get("full_length_leads", []))
    return {
        "formato": (meta or {}).get("lead_layout"),
        "coste_formato": (meta or {}).get("matching_cost"),
        "degradado": bool(calidad.get("degraded")) or pipeline._layout_failed(meta) or plantilla_degradada,
        "compuertas": compuertas,
        "derivaciones_con_senal": list(calidad.get("leads_with_signal", [])),
        "derivaciones_completas": completas,
        "tira_en_II": "II" in completas,
    }


def parse_args(argv: list[str] | None) -> argparse.Namespace:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--digitalizador", required=True, help="Checkout parcheado de Open-ECG-Digitizer.")
    ap.add_argument("--config", required=True, help="Configuración del digitalizador (con su device).")
    ap.add_argument("--imagenes", required=True)
    ap.add_argument("--salida", required=True)
    ap.add_argument("--dispositivo", default="cpu", help="cpu o cuda, para ECGFounder y la sincronización.")
    ap.add_argument("--hilos", type=int, required=True, help="Hilos de torch (torch.set_num_threads).")
    ap.add_argument("--json", required=True)
    ap.add_argument("--pesos", default=None, help="Checkpoint de 1 derivación de ECGFounder; sin él no interpreta.")
    ap.add_argument("--png-muestras", type=int, default=0, help="En cuántas imágenes medir la imagen de depuración.")
    ap.add_argument("--interpretar-extra", default=None, help="Otra carpeta de CSV que interpretar con este dispositivo.")
    return ap.parse_args(argv)


def main(argv: list[str] | None = None) -> dict[str, Any]:
    args = parse_args(argv)
    home = Path(args.digitalizador).resolve()
    imagenes = Path(args.imagenes).resolve()
    salida = Path(args.salida).resolve()
    config = Path(args.config).resolve()
    extra = Path(args.interpretar_extra).resolve() if args.interpretar_extra else None
    cuda = args.dispositivo.startswith("cuda")

    salida.mkdir(parents=True, exist_ok=True)
    # El digitalizador añade filas a digitization_metadata.csv sin borrar las viejas.
    for viejo in list(salida.rglob("digitization_metadata.csv")) + list(salida.rglob(f"*{SUFIJO_CSV}")):
        viejo.unlink()

    datos: dict[str, Any] = {
        "dispositivo": args.dispositivo,
        "salida": str(salida),
        "proceso": {},
        "imagenes": [],
        "ecgfounder": {},
        "observaciones": {},
        "entorno": {},
    }
    proc = datos["proceso"]

    t = time.perf_counter()
    import torch

    proc["Import de torch"] = time.perf_counter() - t
    torch.set_num_threads(args.hilos)

    def sincronizar() -> None:
        if cuda:
            torch.cuda.synchronize()

    @contextlib.contextmanager
    def cronometro(nombre: str, destino: dict[str, float]):
        sincronizar()
        inicio = time.perf_counter()
        try:
            yield
        finally:
            sincronizar()
            destino[nombre] = time.perf_counter() - inicio

    if cuda:
        with cronometro("Inicio de CUDA", proc):
            torch.zeros(1, device=args.dispositivo)

    # Las rutas de la configuración son relativas a la raíz del digitalizador.
    sys.path.insert(0, str(home))
    os.chdir(home)
    with cronometro("Import del digitalizador", proc):
        import src.model.inference_wrapper as inference_wrapper
        from src import digitize as dg
        from src.config.default import get_cfg

    # forward() busca timed_section en el módulo en cada llamada: así sus secciones también
    # sincronizan la GPU.
    inference_wrapper.timed_section = cronometro

    cfg = get_cfg(str(config))
    cfg.merge_from_list(["DATA.images_path", f"{imagenes}/", "DATA.output_path", str(salida)])
    cfg.MODEL.KWARGS.enable_timing = False
    modo_guardado = cfg.DATA.get("save_mode", "all")
    datos["entorno"]["config"] = {
        "archivo": str(config),
        "device_segmentacion": cfg.MODEL.KWARGS.device,
        "device_identificador": cfg.MODEL.KWARGS.config.LAYOUT_IDENTIFIER.KWARGS.device,
        "resample_size": cfg.MODEL.KWARGS.resample_size,
        "save_mode": modo_guardado,
    }

    with cronometro("Carga de los dos U-Net", proc):
        wrapper = dg.import_class_from_path(cfg.MODEL.class_path)(**cfg.MODEL.KWARGS)

    etapa: dict[str, float] = {}
    identificador = wrapper.identifier

    class IdentificadorCronometrado:
        def __call__(self, *a: Any, **k: Any) -> Any:
            with cronometro("Identificación de formato (segundo U-Net)", etapa):
                return identificador(*a, **k)

    wrapper.identifier = IdentificadorCronometrado()

    png_original = dg.save_png_plot
    for nombre_fn, etiqueta in (("save_timeseries_csv", "Guardar CSV"), ("save_matching_cost", "Guardar metadatos")):
        original = getattr(dg, nombre_fn)

        def cronometrada(*a: Any, _original: Any = original, _etiqueta: str = etiqueta, **k: Any) -> Any:
            with cronometro(_etiqueta, etapa):
                return _original(*a, **k)

        setattr(dg, nombre_fn, cronometrada)

    dg.clear_and_prepare_output_dir(cfg)
    archivos = sorted(dg.get_candidate_file_paths(cfg))
    proc["Arranque hasta la primera imagen"] = time.perf_counter() - T_INICIO
    carpeta_png = salida.parent / f"{salida.name}_png_depuracion"

    for orden, ruta in enumerate(archivos):
        etapa.clear()
        sincronizar()
        inicio = time.perf_counter()
        with cronometro("Decodificar imagen", etapa):
            imagen = dg.decode_and_prepare_image(ruta)
            dg.seed_from_image(imagen)
        valores = wrapper(imagen, layout_should_include_substring=None)
        relativa = os.path.relpath(ruta, cfg.DATA.images_path)
        base = os.path.splitext(os.path.join(cfg.DATA.output_path, relativa))[0]
        os.makedirs(os.path.dirname(base), exist_ok=True)
        dg.save_outputs(valores, base, modo_guardado)
        sincronizar()
        total = time.perf_counter() - inicio

        etapas = {ETAPAS_ES.get(k, k): v for k, v in wrapper.times.items()}
        etapas.update(etapa)
        registro: dict[str, Any] = {
            "imagen": Path(relativa).as_posix(),
            "registro": Path(relativa).with_suffix("").as_posix(),
            "orden": orden,
            "tam_entrada_hw": list(imagen.shape[2:]),
            "tam_trabajo_hw": list(valores["input_image"].shape[2:]),
            "total_s": total,
            "etapas": etapas,
            "png_depuracion_s": None,
        }
        if orden < args.png_muestras:
            base_png = carpeta_png / Path(relativa).with_suffix("")
            base_png.parent.mkdir(parents=True, exist_ok=True)
            inicio_png = time.perf_counter()
            png_original(valores, dg.canonical_from_got_values(valores), str(base_png))
            registro["png_depuracion_s"] = time.perf_counter() - inicio_png
        datos["imagenes"].append(registro)
        print(
            json.dumps({"imagen": registro["imagen"], "total_s": round(total, 2), "png_s": registro["png_depuracion_s"]}),
            flush=True,
        )

    # Compuertas de calidad, con el código de ecg_pipeline.
    from ecg_pipeline import pipeline
    from ecg_pipeline.interpret.waveform import assess_quality, load_canonical_csv

    metadatos = pipeline.read_digitization_metadata(salida)
    por_registro = {}
    for csv_path in sorted(salida.rglob(f"*{SUFIJO_CSV}")):
        nombre = pipeline.record_name(csv_path, salida)
        calidad = assess_quality(*load_canonical_csv(str(csv_path)))
        por_registro[nombre] = compuertas_registro(pipeline, metadatos.get(nombre), calidad)
    for registro in datos["imagenes"]:
        registro["csv"] = str(salida / f"{registro['registro']}{SUFIJO_CSV}")
        registro["compuertas"] = por_registro.get(registro["registro"])

    if args.pesos:
        from ecg_pipeline.interpret.interpret_ecg import build_model_for, interpret_csv

        ef = datos["ecgfounder"]
        with cronometro("carga_s", ef):
            modelo, ckpt = build_model_for("rhythm", ckpt_path=args.pesos, device=args.dispositivo)

        def interpretar(carpeta: Path, clave: str) -> None:
            tiempos: dict[str, float] = {}
            observaciones: dict[str, Any] = {}
            for csv_path in sorted(carpeta.rglob(f"*{SUFIJO_CSV}")):
                nombre = pipeline.record_name(csv_path, carpeta)
                medida: dict[str, float] = {}
                try:
                    # k=150: todas las clases, para comparar todas las puntuaciones.
                    with cronometro("s", medida):
                        resultado = interpret_csv(
                            str(csv_path), pathway="rhythm", ckpt=ckpt, k=150, device=args.dispositivo, model=modelo
                        )
                    observaciones[nombre] = resumen_observaciones(resultado)
                except Exception as exc:  # un CSV sin señal utilizable es un resultado, no un fallo del arnés
                    observaciones[nombre] = {"error": f"{type(exc).__name__}: {exc}"}
                tiempos[nombre] = medida.get("s")
            ef[f"inferencia_s_{clave}"] = tiempos
            datos["observaciones"][clave] = observaciones

        interpretar(salida, "propias")
        if extra is not None:
            interpretar(extra, "extra")

    proc["RSS pico MB"] = rss_pico_mb(resource.RUSAGE_SELF)
    proc["Pared total"] = time.perf_counter() - T_INICIO
    entorno = datos["entorno"]
    entorno.update(
        python=sys.version.split()[0],
        torch=torch.__version__,
        cuda_torch=torch.version.cuda,
        hilos_torch=torch.get_num_threads(),
        hilos_interop=torch.get_num_interop_threads(),
        omp_num_threads=os.environ.get("OMP_NUM_THREADS"),
    )
    if cuda:
        proc["GPU memoria pico asignada MB"] = torch.cuda.max_memory_allocated() / (1 << 20)
        proc["GPU memoria pico reservada MB"] = torch.cuda.max_memory_reserved() / (1 << 20)
        entorno["gpu"] = torch.cuda.get_device_name(0)
        entorno["cudnn"] = torch.backends.cudnn.version()

    datos = limpiar_json(datos)
    Path(args.json).write_text(json.dumps(datos, indent=2, ensure_ascii=False))
    print(json.dumps({k: (round(v, 2) if isinstance(v, float) else v) for k, v in proc.items()}), flush=True)
    return datos


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/medir/medir_estudios.py
"""Estudios seguidos, uno por llamada, como los procesa el worker de api-EKG.

Por estudio, lo mismo que analysis/runner.py de api-EKG: pipeline.run(skip_interpretation=True)
para digitalizar y pasar las compuertas, y después interpret_csv con el modelo de ECGFounder
cargado una sola vez, en el primer estudio. El modo del digitalizador llega por
ECG_DIGITIZER_MODE (subprocess o persistent) y los hilos por ECG_TORCH_THREADS, como en
producción.

El primer estudio paga el arranque en frío: imports, inicio de CUDA, carga de los modelos del
digitalizador y de ECGFounder. Para separarlo de lo que varía de una imagen a otra, la
secuencia repite al final la primera imagen, y el sobrecoste del arranque es el primer
estudio menos esa repetición.

    python medir_estudios.py --imagenes A.png B.png A.png --salida DIR --config CFG \
        --dispositivo cpu|cuda --pesos CKPT --json FILE
"""

from __future__ import annotations

import argparse
import json
import math
import resource
import shutil
import sys
import tempfile
import time
from pathlib import Path
from typing import Any

T_INICIO = time.perf_counter()


def limpiar_json(valor: Any) -> Any:
    if isinstance(valor, float) and not math.isfinite(valor):
        return None
    if isinstance(valor, dict):
        return {str(k): limpiar_json(v) for k, v in valor.items()}
    if isinstance(valor, (list, tuple)):
        return [limpiar_json(v) for v in valor]
    return valor


def rss_pico_mb(quien: int) -> float:
    rss = resource.getrusage(quien).ru_maxrss
    return rss / (1 << 20) if sys.platform == "darwin" else rss / 1024


def main(argv: list[str] | None = None) -> dict[str, Any]:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--imagenes", nargs="+", required=True, help="Una imagen por estudio, en orden.")
    ap.add_argument("--salida", required=True)
    ap.add_argument("--config", required=True, help="Configuración del digitalizador (con su device).")
    ap.add_argument("--dispositivo", default="cpu", help="Dispositivo de ECGFounder.")
    ap.add_argument("--pesos", required=True, help="Checkpoint de 1 derivación de ECGFounder.")
    ap.add_argument("--json", required=True)
    args = ap.parse_args(argv)

    salida = Path(args.salida).resolve()
    salida.mkdir(parents=True, exist_ok=True)

    t = time.perf_counter()
    from ecg_pipeline import digitizer, pipeline, threads

    imports_pipeline = time.perf_counter() - t

    modelo: Any = None
    ckpt = ""
    interpret_csv: Any = None
    estudios: list[dict[str, Any]] = []
    for orden, imagen in enumerate(args.imagenes, start=1):
        imagen = Path(imagen).resolve()
        carpeta = salida / f"estudio_{orden:02d}"
        if carpeta.exists():
            shutil.rmtree(carpeta)

        inicio = time.perf_counter()
        with tempfile.TemporaryDirectory(prefix="estudio-") as staged:
            shutil.copy(imagen, Path(staged) / imagen.name)
            resultados = pipeline.run(
                image_dir=staged,
                output_dir=carpeta,
                config=args.config,
                skip_interpretation=True,
                upscale="auto",
                quiet=True,
            )
        digitalizacion = time.perf_counter() - inicio
        registro = resultados[0] if resultados else {}

        inicio = time.perf_counter()
        carga = 0.0
        if modelo is None:
            from ecg_pipeline.interpret.interpret_ecg import build_model_for, interpret_csv

            modelo, ckpt = build_model_for("rhythm", ckpt_path=args.pesos, device=args.dispositivo)
            carga = time.perf_counter() - inicio
        top1, error = None, None
        if registro.get("source_csv"):
            try:
                resultado = interpret_csv(
                    registro["source_csv"], pathway="rhythm", ckpt=ckpt, k=10, device=args.dispositivo, model=modelo
                )
                top1 = resultado["topk"][0]["label"] if resultado.get("topk") else None
            except Exception as exc:
                error = f"{type(exc).__name__}: {exc}"
        interpretacion = time.perf_counter() - inicio

        estudios.append(
            {
                "orden": orden,
                "imagen": str(imagen),
                "digitalizacion_s": digitalizacion,
                "interpretacion_s": interpretacion,
                "carga_ecgfounder_s": carga,
                "total_s": digitalizacion + interpretacion,
                "fin_desde_inicio_proceso_s": time.perf_counter() - T_INICIO,
                "formato": (registro.get("digitization") or {}).get("lead_layout"),
                "degradado": registro.get("degraded"),
                "compuertas": registro.get("gates"),
                "top1": top1,
                "error": error or registro.get("error"),
                "csv": registro.get("source_csv"),
            }
        )
        print(json.dumps({k: estudios[-1][k] for k in ("orden", "total_s", "formato", "top1")}), flush=True)

    persistente = digitizer._persistent
    carga_persistente = persistente.load_seconds if persistente is not None else None
    arranques_persistente = persistente.starts if persistente is not None else None
    digitizer.close_persistent_digitizer()

    datos: dict[str, Any] = {
        "modo": digitizer.resolve_mode(),
        "dispositivo": args.dispositivo,
        "config": str(Path(args.config).resolve()),
        "hilos": threads.resolve_threads(),
        "imports_pipeline_s": imports_pipeline,
        "inicio_proceso_hasta_primer_estudio_s": estudios[0]["fin_desde_inicio_proceso_s"] - estudios[0]["total_s"],
        "carga_digitalizador_persistente_s": carga_persistente,
        "arranques_digitalizador_persistente": arranques_persistente,
        "estudios": estudios,
        "pared_total_s": time.perf_counter() - T_INICIO,
        "rss_pico_proceso_mb": rss_pico_mb(resource.RUSAGE_SELF),
        "rss_pico_digitalizador_mb": rss_pico_mb(resource.RUSAGE_CHILDREN),
    }
    if args.dispositivo.startswith("cuda") and modelo is not None:
        import torch

        datos["gpu_memoria_pico_ecgfounder_mb"] = torch.cuda.max_memory_allocated() / (1 << 20)

    datos = limpiar_json(datos)
    Path(args.json).write_text(json.dumps(datos, indent=2, ensure_ascii=False))
    return datos


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/medir/calidad.py
"""Calidad de la digitalización frente a la señal WFDB original, y CPU frente a GPU.

Métricas del estudio de escalas (docs/RENDIMIENTO.md, sección 2) y de
scripts/validate_against_wfdb.py de la rama validacion: por derivación, correlación r y SNR
sobre el tramo solapado, tras buscar un desfase de hasta 50 ms. El SNR principal se calcula
con la media de cada traza restada; el SNR sin restarla se guarda también. Por registro,
mediana sobre las derivaciones; por variante, media de esas medianas.

Tolerancia, la misma del estudio de escalas: la GPU se acepta solo si, frente a la CPU medida
aquí mismo con las mismas imágenes,
  - el SNR medio no empeora más de 0,5 dB en ninguna variante,
  - la r media no empeora más de 0,002 en ninguna variante,
  - no aparece ninguna compuerta nueva, ningún formato distinto y ninguna tira fuera de II.

Lee los JSON de medir_etapas.py (y, si se pasan, los de medir_estudios.py). Solo necesita
numpy, y wfdb para leer la señal original.

    python calidad.py --registros REG.json --etapas-cpu CPU.json [--etapas-gpu GPU.json] \
        --wfdb-root DIR --json SALIDA.json [--estudios E1.json E2.json ...]
"""

from __future__ import annotations

import argparse
import json
import math
import statistics
from pathlib import Path
from typing import Any

import numpy as np

CANONICAL_LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
CANONICAL_N = 5000  # 10 s a 500 Hz
MAX_LAG_SAMPLES = 25  # +-50 ms
SUFIJO_CSV = "_timeseries_canonical.csv"

TOLERANCIA = {"snr_db": 0.5, "r": 0.002}

# Línea base del estudio de escalas (resample_size 3000, MacBook Pro 2019, CPU), por registro:
# SNR con la media restada (dB) y r, medianas sobre las derivaciones. Solo como referencia:
# la tolerancia se evalúa GPU frente a CPU medidas en el mismo entorno.
REFERENCIA_MAC = {
    ("limpio", "1172"): (16.9, 0.991),
    ("limpio", "1355"): (16.9, 0.990),
    ("limpio", "8645"): (17.3, 0.991),
    ("limpio", "7221"): (16.3, 0.989),
    ("limpio", "8215"): (17.7, 0.992),
    ("limpio", "17386"): (17.9, 0.993),
    ("limpio", "17690"): (20.3, 0.995),
    ("limpio", "2433"): (20.4, 0.996),
    ("limpio", "6231"): (18.7, 0.994),
    ("limpio", "18506"): (20.3, 0.995),
    ("limpio", "19424"): (19.4, 0.995),
    ("limpio", "8507"): (19.0, 0.994),
    ("distorsionado", "1172"): (15.4, 0.987),
    ("distorsionado", "1355"): (13.6, 0.980),
    ("distorsionado", "8645"): (14.6, 0.984),
    ("distorsionado", "7221"): (15.3, 0.986),
    ("distorsionado", "8215"): (13.7, 0.979),
    ("distorsionado", "17386"): (14.9, 0.986),
    ("distorsionado", "17690"): (18.9, 0.994),
    ("distorsionado", "2433"): (16.9, 0.990),
    ("distorsionado", "6231"): (14.8, 0.986),
    ("distorsionado", "18506"): (17.6, 0.992),
    ("distorsionado", "19424"): (17.1, 0.991),
    ("distorsionado", "8507"): (16.6, 0.989),
}


# ----------------------------------------------------------------- señal frente a WFDB


def load_canonical_csv(path: str | Path) -> tuple[np.ndarray, list[str]]:
    """(derivaciones, muestras) y los nombres de columna, como waveform.load_canonical_csv."""
    with open(path) as fh:
        header = fh.readline().strip().split(",")
    data = np.genfromtxt(path, delimiter=",", skip_header=1)
    if data.ndim == 1:
        data = data[:, None]
    return data.T, header


def load_reference_uv(wfdb_root: Path, filename_hr: str) -> np.ndarray:
    """Registro WFDB como (12, N) en microvoltios, en el orden canónico."""
    import wfdb

    record = wfdb.rdrecord(str(wfdb_root / filename_hr))
    sig_mv = record.p_signal.T
    indices = {n.strip().upper(): i for i, n in enumerate(record.sig_name)}
    out = np.full((len(CANONICAL_LEADS), sig_mv.shape[1]), np.nan)
    for i, lead in enumerate(CANONICAL_LEADS):
        idx = indices.get(lead.upper())
        if idx is not None:
            out[i] = sig_mv[idx] * 1000.0
    return out


def _mascara_desfase(d: np.ndarray, ref: np.ndarray, shift: int) -> np.ndarray:
    mask = ~np.isnan(d) & ~np.isnan(ref)
    if shift > 0:
        mask[:shift] = False
    elif shift < 0:
        mask[shift:] = False
    return mask


def best_lag_align(dig: np.ndarray, ref: np.ndarray, max_lag: int = MAX_LAG_SAMPLES) -> int:
    """Desfase en muestras, dentro de +-max_lag, que maximiza |r| sobre el solape."""
    best_shift, best_abs_r = 0, -1.0
    for shift in range(-max_lag, max_lag + 1):
        d = np.roll(dig, shift)
        mask = _mascara_desfase(d, ref, shift)
        if mask.sum() < 30:
            continue
        dv, rv = d[mask], ref[mask]
        if np.std(dv) < 1e-9 or np.std(rv) < 1e-9:
            continue
        r = float(np.corrcoef(dv, rv)[0, 1])
        if abs(r) > best_abs_r:
            best_abs_r, best_shift = abs(r), shift
    return best_shift


def lead_metrics(dig: np.ndarray, ref: np.ndarray) -> dict[str, Any] | None:
    """r, SNR con y sin la media restada, sobre el solape alineado. None si no hay solape."""
    if (~np.isnan(dig) & ~np.isnan(ref)).sum() < 30:
        return None
    shift = best_lag_align(dig, ref)
    d = np.roll(dig, shift)
    mask = _mascara_desfase(d, ref, shift)
    if mask.sum() < 30:
        return None
    dv, rv = d[mask], ref[mask]
    r = float("nan") if np.std(dv) < 1e-9 or np.std(rv) < 1e-9 else float(np.corrcoef(dv, rv)[0, 1])
    err = float(np.mean((dv - rv) ** 2))
    snr = 10 * math.log10(float(np.mean(rv**2)) / err) if err > 0 else float("inf")
    dc, rc = dv - dv.mean(), rv - rv.mean()
    err_c = float(np.mean((dc - rc) ** 2))
    snr_c = 10 * math.log10(float(np.mean(rc**2)) / err_c) if err_c > 0 else float("inf")
    return {"r": r, "snr_db": snr_c, "snr_sin_centrar_db": snr, "desfase_muestras": shift}


def _mediana(valores: list[float]) -> float:
    finitos = [v for v in valores if v is not None and math.isfinite(v)]
    return statistics.median(finitos) if finitos else float("nan")


def _media(valores: list[float]) -> float:
    finitos = [v for v in valores if v is not None and math.isfinite(v)]
    return statistics.fmean(finitos) if finitos else float("nan")


def metricas_registro(csv_path: str | Path, ref: np.ndarray) -> dict[str, Any]:
    """Medianas sobre las derivaciones con solape, y el detalle por derivación."""
    if not Path(csv_path).is_file():
        return {"r": float("nan"), "snr_db": float("nan"), "snr_sin_centrar_db": float("nan"), "derivaciones": {}}
    dig, nombres = load_canonical_csv(csv_path)
    por_derivacion = {}
    for i, lead in enumerate(CANONICAL_LEADS):
        if lead not in nombres:
            continue
        d = dig[nombres.index(lead)].astype(np.float64)
        n = min(d.size, ref.shape[1], CANONICAL_N)
        m = lead_metrics(d[:n], ref[i, :n].astype(np.float64))
        if m is not None:
            por_derivacion[lead] = m
    return {
        "r": _mediana([m["r"] for m in por_derivacion.values()]),
        "snr_db": _mediana([m["snr_db"] for m in por_derivacion.values()]),
        "snr_sin_centrar_db": _mediana([m["snr_sin_centrar_db"] for m in por_derivacion.values()]),
        "derivaciones": por_derivacion,
    }


# ----------------------------------------------------------------- CPU frente a GPU


def diferencia_csv(a: str | Path, b: str | Path) -> dict[str, Any]:
    """Diferencia máxima absoluta (µV) entre dos CSV de señal, y huecos que no coinciden."""
    if not (Path(a).is_file() and Path(b).is_file()):
        return {"mismo_tamano": False, "max_abs_uv": None, "huecos_distintos": None, "identicos": False}
    x, nx = load_canonical_csv(a)
    y, ny = load_canonical_csv(b)
    if x.shape != y.shape or nx != ny:
        return {"mismo_tamano": False, "max_abs_uv": None, "huecos_distintos": None, "identicos": False}
    nan_x, nan_y = np.isnan(x), np.isnan(y)
    ambos = ~nan_x & ~nan_y
    max_abs = float(np.max(np.abs(x[ambos] - y[ambos]))) if ambos.any() else 0.0
    distintos = int(np.sum(nan_x != nan_y))
    return {
        "mismo_tamano": True,
        "max_abs_uv": max_abs,
        "huecos_distintos": distintos,
        "identicos": max_abs == 0.0 and distintos == 0,
    }


def comparar_observaciones(a: dict[str, Any] | None, b: dict[str, Any] | None) -> dict[str, Any]:
    """Si ECGFounder dijo lo mismo: etiquetas del top 10, top 3, marcadas, y puntuaciones."""
    if not a or not b or a.get("error") or b.get("error") or not a.get("topk") or not b.get("topk"):
        return {
            "comparable": False,
            "top10_iguales": None,
            "top3_iguales": None,
            "marcadas_iguales": None,
            "max_dif_puntuacion": None,
            "error_a": (a or {}).get("error"),
            "error_b": (b or {}).get("error"),
        }
    top_a = [r["label"] for r in a["topk"]]
    top_b = [r["label"] for r in b["topk"]]
    prob_a = {r["label"]: r["prob"] for r in a["topk"]}
    prob_b = {r["label"]: r["prob"] for r in b["topk"]}
    comunes = set(prob_a) & set(prob_b)
    marcadas_a = sorted(r["label"] for r in (a.get("marcadas") or []))
    marcadas_b = sorted(r["label"] for r in (b.get("marcadas") or []))
    return {
        "comparable": True,
        "top10_iguales": top_a[:10] == top_b[:10],
        "top3_iguales": top_a[:3] == top_b[:3],
        "marcadas_iguales": marcadas_a == marcadas_b,
        "max_dif_puntuacion": max((abs(prob_a[k] - prob_b[k]) for k in comunes), default=None),
        "derivaciones_ritmo_iguales": a.get("derivaciones_ritmo") == b.get("derivaciones_ritmo"),
    }


def resumen_etapas(etapas: dict[str, Any], wfdb_root: Path, registros: dict[str, dict[str, Any]]) -> dict:
    """Por (variante, ecg_id): métricas contra WFDB, compuertas y CSV, de una corrida de medir_etapas."""
    out = {}
    observaciones = (etapas.get("observaciones") or {}).get("propias") or {}
    for imagen in etapas["imagenes"]:
        variante, ecg_id = Path(imagen["registro"]).parent.name, Path(imagen["registro"]).name
        info = registros[ecg_id]
        ref = load_reference_uv(wfdb_root, info["filename_hr"])
        metricas = metricas_registro(imagen["csv"], ref)
        compuertas = imagen.get("compuertas") or {}
        out[(variante, ecg_id)] = {
            "variante": variante,
            "ecg_id": ecg_id,
            "grupo": info.get("grupo"),
            "csv": imagen["csv"],
            "r": metricas["r"],
            "snr_db": metricas["snr_db"],
            "snr_sin_centrar_db": metricas["snr_sin_centrar_db"],
            "n_derivaciones": len(metricas["derivaciones"]),
            "formato": compuertas.get("formato"),
            "tira_en_II": compuertas.get("tira_en_II"),
            "degradado": compuertas.get("degradado"),
            "compuertas": compuertas.get("compuertas") or [],
            "observaciones": observaciones.get(imagen["registro"]),
        }
    return out


def comparar(cpu: dict, gpu: dict | None, extra_gpu: dict[str, Any] | None = None) -> dict[str, Any]:
    """Tabla por registro y evaluación de la tolerancia por variante."""
    filas = []
    for clave in sorted(cpu, key=lambda k: (k[0] != "limpio", k[0], int(k[1]) if k[1].isdigit() else k[1])):
        c = cpu[clave]
        mac = REFERENCIA_MAC.get(clave, (None, None))
        fila: dict[str, Any] = {
            "variante": c["variante"],
            "ecg_id": c["ecg_id"],
            "grupo": c["grupo"],
            "snr_mac_db": mac[0],
            "r_mac": mac[1],
            "snr_cpu_db": c["snr_db"],
            "r_cpu": c["r"],
            "snr_sin_centrar_cpu_db": c["snr_sin_centrar_db"],
            "formato_cpu": c["formato"],
            "tira_en_II_cpu": c["tira_en_II"],
            "compuertas_cpu": ";".join(c["compuertas"]),
        }
        g = (gpu or {}).get(clave)
        if g is not None:
            nuevas = sorted(set(g["compuertas"]) - set(c["compuertas"]))
            obs = comparar_observaciones(c["observaciones"], g["observaciones"])
            registro = f"{c['variante']}/{c['ecg_id']}"
            mismo_csv = comparar_observaciones(c["observaciones"], (extra_gpu or {}).get(registro))
            fila.update(
                {
                    "snr_gpu_db": g["snr_db"],
                    "r_gpu": g["r"],
                    "snr_sin_centrar_gpu_db": g["snr_sin_centrar_db"],
                    "dif_snr_db": g["snr_db"] - c["snr_db"],
                    "dif_r": g["r"] - c["r"],
                    "formato_gpu": g["formato"],
                    "tira_en_II_gpu": g["tira_en_II"],
                    "compuertas_gpu": ";".join(g["compuertas"]),
                    "mismo_formato": g["formato"] == c["formato"],
                    "compuertas_nuevas": ";".join(nuevas),
                    "tira_fuera_de_II_nueva": bool(c["tira_en_II"]) and not g["tira_en_II"],
                    **{f"csv_{k}": v for k, v in diferencia_csv(c["csv"], g["csv"]).items()},
                    "obs_top10_iguales": obs["top10_iguales"],
                    "obs_top3_iguales": obs["top3_iguales"],
                    "obs_marcadas_iguales": obs["marcadas_iguales"],
                    "obs_max_dif_puntuacion": obs["max_dif_puntuacion"],
                    "ecgfounder_mismo_csv_top10_iguales": mismo_csv["top10_iguales"],
                    "ecgfounder_mismo_csv_marcadas_iguales": mismo_csv["marcadas_iguales"],
                    "ecgfounder_mismo_csv_max_dif_puntuacion": mismo_csv["max_dif_puntuacion"],
                }
            )
        filas.append(fila)
    return {"tolerancia": TOLERANCIA, "por_registro": filas, **evaluar(filas, gpu is not None)}


def evaluar(filas: list[dict[str, Any]], hay_gpu: bool) -> dict[str, Any]:
    por_variante: dict[str, Any] = {}
    incumplimientos: list[str] = []
    for variante in sorted({f["variante"] for f in filas}):
        fv = [f for f in filas if f["variante"] == variante]
        v: dict[str, Any] = {
            "registros": len(fv),
            "snr_cpu_db": _media([f["snr_cpu_db"] for f in fv]),
            "r_cpu": _media([f["r_cpu"] for f in fv]),
            "tiras_en_II_cpu": sum(bool(f["tira_en_II_cpu"]) for f in fv),
            "degradados_cpu": sum(bool(f["compuertas_cpu"]) for f in fv),
        }
        if hay_gpu:
            v.update(
                {
                    "snr_gpu_db": _media([f.get("snr_gpu_db") for f in fv]),
                    "r_gpu": _media([f.get("r_gpu") for f in fv]),
                    "tiras_en_II_gpu": sum(bool(f.get("tira_en_II_gpu")) for f in fv),
                    "degradados_gpu": sum(bool(f.get("compuertas_gpu")) for f in fv),
                    "formatos_distintos": sum(f.get("mismo_formato") is False for f in fv),
                    "compuertas_nuevas": sum(bool(f.get("compuertas_nuevas")) for f in fv),
                    "tiras_fuera_de_II_nuevas": sum(bool(f.get("tira_fuera_de_II_nueva")) for f in fv),
                    "csv_identicos": sum(bool(f.get("csv_identicos")) for f in fv),
                    "csv_max_abs_uv": max((f["csv_max_abs_uv"] for f in fv if f.get("csv_max_abs_uv") is not None), default=None),
                    "obs_marcadas_iguales": sum(bool(f.get("obs_marcadas_iguales")) for f in fv),
                    "obs_top3_iguales": sum(bool(f.get("obs_top3_iguales")) for f in fv),
                    "obs_top10_iguales": sum(bool(f.get("obs_top10_iguales")) for f in fv),
                }
            )
            v["dif_snr_db"] = v["snr_gpu_db"] - v["snr_cpu_db"]
            v["dif_r"] = v["r_gpu"] - v["r_cpu"]
            v["cumple_snr"] = bool(v["dif_snr_db"] >= -TOLERANCIA["snr_db"])
            v["cumple_r"] = bool(v["dif_r"] >= -TOLERANCIA["r"])
            v["cumple_compuertas"] = v["formatos_distintos"] == 0 and v["compuertas_nuevas"] == 0
            v["cumple_compuertas"] = v["cumple_compuertas"] and v["tiras_fuera_de_II_nuevas"] == 0
            if not v["cumple_snr"]:
                incumplimientos.append(f"{variante}: el SNR medio cae {-v['dif_snr_db']:.2f} dB (tolerancia 0,5)")
            if not v["cumple_r"]:
                incumplimientos.append(f"{variante}: la r media cae {-v['dif_r']:.4f} (tolerancia 0,002)")
            for f in fv:
                if f.get("mismo_formato") is False:
                    incumplimientos.append(f"{variante} {f['ecg_id']}: formato {f['formato_gpu']} en vez de {f['formato_cpu']}")
                if f.get("compuertas_nuevas"):
                    incumplimientos.append(f"{variante} {f['ecg_id']}: compuerta nueva {f['compuertas_nuevas']}")
                if f.get("tira_fuera_de_II_nueva"):
                    incumplimientos.append(f"{variante} {f['ecg_id']}: la tira de ritmo deja de estar en II")
        por_variante[variante] = v
    return {
        "por_variante": por_variante,
        "cumple": (not incumplimientos) if hay_gpu else None,
        "incumplimientos": incumplimientos,
    }


def consistencia_estudios(estudios: list[dict[str, Any]], etapas: dict[str, dict[str, Any]]) -> list[dict[str, Any]]:
    """El CSV de cada estudio (pipeline.run) frente al del perfil por etapas, mismo dispositivo.

    Comprueba que el arnés por etapas digitaliza igual que el pipeline, como en RENDIMIENTO.md.
    """
    filas = []
    for corrida in estudios:
        perfil = etapas.get(corrida["dispositivo"])
        if perfil is None:
            continue
        csv_por_imagen = {im["imagen"]: im["csv"] for im in perfil["imagenes"]}
        for e in corrida["estudios"]:
            relativa = "/".join(Path(e["imagen"]).parts[-2:])
            referencia = csv_por_imagen.get(relativa)
            if referencia is None or not e.get("csv"):
                continue
            filas.append(
                {
                    "dispositivo": corrida["dispositivo"],
                    "modo": corrida["modo"],
                    "orden": e["orden"],
                    "imagen": relativa,
                    **diferencia_csv(referencia, e["csv"]),
                }
            )
    return filas


def limpiar_json(valor: Any) -> Any:
    if isinstance(valor, float) and not math.isfinite(valor):
        return None
    if isinstance(valor, dict):
        return {str(k): limpiar_json(v) for k, v in valor.items()}
    if isinstance(valor, (list, tuple)):
        return [limpiar_json(v) for v in valor]
    return valor


def main(argv: list[str] | None = None) -> dict[str, Any]:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--registros", required=True, help="JSON {ecg_id: {grupo, filename_hr}}.")
    ap.add_argument("--etapas-cpu", required=True)
    ap.add_argument("--etapas-gpu", default=None)
    ap.add_argument("--estudios", nargs="*", default=[])
    ap.add_argument("--wfdb-root", required=True)
    ap.add_argument("--json", required=True)
    args = ap.parse_args(argv)

    registros = json.loads(Path(args.registros).read_text())
    wfdb_root = Path(args.wfdb_root)
    etapas_cpu = json.loads(Path(args.etapas_cpu).read_text())
    etapas_gpu = json.loads(Path(args.etapas_gpu).read_text()) if args.etapas_gpu else None

    cpu = resumen_etapas(etapas_cpu, wfdb_root, registros)
    gpu = resumen_etapas(etapas_gpu, wfdb_root, registros) if etapas_gpu else None
    extra = ((etapas_gpu or {}).get("observaciones") or {}).get("extra")
    resultado = comparar(cpu, gpu, extra)

    estudios = [json.loads(Path(p).read_text()) for p in args.estudios]
    perfiles = {"cpu": etapas_cpu, **({etapas_gpu["dispositivo"]: etapas_gpu} if etapas_gpu else {})}
    resultado["consistencia_estudios"] = consistencia_estudios(estudios, perfiles)

    resultado = limpiar_json(resultado)
    Path(args.json).write_text(json.dumps(resultado, indent=2, ensure_ascii=False))
    for variante, v in resultado["por_variante"].items():
        print(variante, json.dumps(v, ensure_ascii=False))
    print("cumple:", resultado["cumple"], resultado["incumplimientos"])
    return resultado


if __name__ == "__main__":
    main()

## 3. Imágenes de validación

Se bajan de PhysioNet los 24 registros de la validación (PTB-XL 1.0.3, fold 10, CC BY 4.0), unos 3 MB, y ECG-Image-Kit los dibuja como un electrocardiograma en papel con los mismos comandos del estudio de escalas: semilla 42, 200 dpi, formato 3x4 con tira de ritmo en II y sin pulso de calibración. La versión distorsionada añade rotación, ruido, recorte, temperatura de color y arrugas. Cada imagen mide 2200x1700.

Solo se miden los registros elegidos en el estudio de escalas: NORM 1172, 1355 y 8645; AFIB 7221, 8215 y 17386; CRBBB/CLBBB 17690 y 2433; IMI/AMI 6231 y 18506; SBRAD/STACH 19424 y 8507.

Las imágenes no serán idénticas píxel a píxel a las del Mac: el ruido de la distorsión no tiene semilla y el orden de los archivos cambia entre sistemas. No importa para la decisión, porque CPU y GPU se comparan sobre las mismas imágenes generadas aquí. Las cifras del Mac aparecen solo como referencia.

In [ ]:
import urllib.request

from IPython.display import display
from PIL import Image

BASE_PHYSIONET = "https://physionet.org/files/ptb-xl/1.0.3/"


def ruta_wfdb(ecg_id):
    n = int(ecg_id)
    return f"records500/{n // 1000 * 1000:05d}/{n:05d}_hr"


for ecg_id in REGISTROS_VALIDACION:
    for extension in (".hea", ".dat"):
        destino = PTBXL / (ruta_wfdb(ecg_id) + extension)
        if destino.is_file() and destino.stat().st_size > 0:
            continue
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(BASE_PHYSIONET + ruta_wfdb(ecg_id) + extension, destino)
print(f"PTB-XL: {len(REGISTROS_VALIDACION)} registros en {PTBXL}")

COMUN = ["-se", "42", "-r", "200", "--num_columns", "-1", "--full_mode", "II", "--calibration_pulse", "0"]
DISTORSION = [
    "--store_config", "1", "--augment", "-rot", "8", "-noise", "30", "-c", "0.01", "-t", "40000",
    "--deterministic_rot", "--deterministic_noise", "--deterministic_crop", "--deterministic_temp",
    "--wrinkles", "-ca", "15", "-nv", "3", "-nh", "3",
    "--deterministic_angle", "--deterministic_vertical", "--deterministic_horizontal",
]  # fmt: skip
GENERADOR = KIT / "codes" / "ecg-image-generator"

for variante, extra in (("limpio", []), ("distorsionado", DISTORSION)):
    carpeta = IMAGENES / variante
    carpeta.mkdir(parents=True, exist_ok=True)
    # Fuera lo que no toca medir, por si MODO_RAPIDO cambió desde la última ejecución.
    for sobrante in carpeta.glob("*.png"):
        if sobrante.stem not in REGISTROS:
            sobrante.unlink()
    if all((carpeta / f"{i}.png").is_file() for i in REGISTROS):
        print(f"{variante}: imágenes ya generadas")
        continue
    generadas = VAL / f"generadas_{variante}"
    shutil.rmtree(generadas, ignore_errors=True)
    inicio = time.perf_counter()
    cmd = [str(VENV_KIT_PY), str(MEDIR / "generar_kit.py"), str(GENERADOR), "-i", str(PTBXL / "records500")]
    r = subprocess.run(cmd + ["-o", str(generadas), *COMUN, *extra], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"ECG-Image-Kit falló generando las imágenes {variante}.")
    for ecg_id in REGISTROS:
        shutil.copy(generadas / (ruta_wfdb(ecg_id).split("/", 1)[1] + "-0.png"), carpeta / f"{ecg_id}.png")
    print(f"{variante}: {len(REGISTROS_VALIDACION)} imágenes dibujadas en {time.perf_counter() - inicio:.0f} s")

for referencia in SECUENCIA_ESTUDIOS:
    assert (IMAGENES / referencia).is_file(), f"SECUENCIA_ESTUDIOS pide {referencia}, que no está entre las imágenes"

muestra = next(iter(REGISTROS))
tira = Image.new("RGB", (1100, 425), "white")
for k, variante in enumerate(VARIANTES):
    imagen = Image.open(IMAGENES / variante / f"{muestra}.png")
    print(f"{variante}/{muestra}.png: {imagen.size[0]}x{imagen.size[1]}")
    imagen = imagen.convert("RGB")
    imagen.thumbnail((550, 425))
    tira.paste(imagen, (k * 550, 0))
display(tira)

## 4. Medir en CPU y en GPU

Cómo se elige el dispositivo, sin cambiar nada del pipeline:

- Digitalizador: Open-ECG-Digitizer no tiene una configuración especial de GPU. Basta con cambiar `device` en las dos claves de su configuración que lo llevan, `MODEL.KWARGS.device` (la red de segmentación) y `MODEL.KWARGS.config.LAYOUT_IDENTIFIER.KWARGS.device` (la red que identifica el formato). El notebook copia `configs/digitizer_cpu.yml` y cambia esas dos líneas; el pipeline recibe la copia con `pipeline.run(config=...)`. El resto del código del digitalizador ya mueve a CPU lo que trabaja con numpy.
- ECGFounder: el parámetro `device` de `build_model_for` e `interpret_csv`, el mismo que api-EKG toma de `ECG_DEVICE`.

En las mediciones de CPU se oculta la GPU (`CUDA_VISIBLE_DEVICES` vacío) para que nada la use por accidente.

La imagen de depuración (un PNG que dibuja el digitalizador y que no lee nadie) se desactiva con `DATA.save_mode: timeseries_only`, que no toca el CSV ni los metadatos. Su coste se mide aparte en las primeras imágenes.

In [ ]:
import threading
from statistics import fmean

import pandas as pd

NOMBRE = {"cpu": f"CPU ({HILOS_CPU} hilos)", "cuda": f"GPU ({GPU_NOMBRE})"}


def media(valores):
    valores = [v for v in valores if v is not None]
    return fmean(valores) if valores else None


def config_digitalizador(dispositivo):
    """configs/digitizer_cpu.yml con otro device y sin imagen de depuración."""
    texto = (REPO / "configs" / "digitizer_cpu.yml").read_text()
    assert texto.count("device: 'cpu'") == 2, "configs/digitizer_cpu.yml ya no tiene las dos claves device esperadas"
    assert texto.count("save_mode: 'all'") == 1, "configs/digitizer_cpu.yml ya no tiene save_mode: 'all'"
    texto = texto.replace("device: 'cpu'", f"device: '{dispositivo}'")
    texto = texto.replace("save_mode: 'all'", "save_mode: 'timeseries_only'")
    ruta = MEDIR / f"digitizer_{dispositivo}.yml"
    ruta.write_text(texto)
    return ruta


def entorno_medicion(dispositivo, modo=None):
    """Variables de entorno de una medición: checkout, pesos, umbrales, hilos y modo."""
    env = dict(os.environ)
    hilos = str(HILOS_CPU)
    env.update(
        {
            "OPEN_ECG_DIGITIZER_HOME": str(DIGITALIZADOR),
            "ECGFOUNDER_WEIGHTS_DIR": str(REPO / "weights"),
            "ECGFOUNDER_THRESHOLDS_DIR": str(UMBRALES),
            "ECG_TORCH_THREADS": hilos,
            "OMP_NUM_THREADS": hilos,
            "MKL_NUM_THREADS": hilos,
            "PYTHONUNBUFFERED": "1",
        }
    )
    env.pop("ECG_DIGITIZER_RESAMPLE_SIZE", None)  # resolución de trabajo por defecto, la validada
    if modo:
        env["ECG_DIGITIZER_MODE"] = modo
    if dispositivo == "cpu":
        env["CUDA_VISIBLE_DEVICES"] = ""
    return env


class MuestreoGPU:
    """Memoria usada en la GPU según nvidia-smi, cada medio segundo, mientras dura el bloque.

    Incluye todo lo que hay en la GPU: los modelos, las activaciones y el contexto de CUDA de
    cada proceso (el digitalizador y ECGFounder van en procesos distintos).
    """

    def __init__(self):
        self.base_mb = None
        self.pico_mb = None

    def __enter__(self):
        if HAY_GPU:
            texto = salida_de(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"])
            self.base_mb = self.pico_mb = float(texto.splitlines()[0]) if texto else None
            self._proc = subprocess.Popen(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-lms", "500"],
                stdout=subprocess.PIPE,
                text=True,
            )
            self._hilo = threading.Thread(target=self._leer, daemon=True)
            self._hilo.start()
        return self

    def _leer(self):
        for linea in self._proc.stdout:
            try:
                self.pico_mb = max(self.pico_mb or 0.0, float(linea.strip().splitlines()[0]))
            except (ValueError, IndexError):
                pass

    def __exit__(self, *exc):
        if HAY_GPU:
            self._proc.terminate()
            self._proc.wait()
            self._hilo.join(timeout=5)
        return False


def lanzar(nombre, cmd, env):
    """Ejecuta una medición en el entorno del pipeline, enseña su progreso y guarda el log."""
    log = RESULTADOS / "logs" / f"{nombre}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    print(f"==> {nombre}", flush=True)
    inicio = time.perf_counter()
    with MuestreoGPU() as gpu, open(log, "w") as fh:
        proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=MEDIR)
        for linea in proc.stdout:
            fh.write(linea)
            if linea.startswith("{"):
                print("   ", linea.rstrip(), flush=True)
        proc.wait()
    pared = time.perf_counter() - inicio
    if proc.returncode != 0:
        print(log.read_text()[-4000:])
        raise RuntimeError(f"{nombre} terminó con código {proc.returncode}; log completo en {log}")
    print(f"    {nombre}: {pared / 60:.1f} min", flush=True)
    return {"pared_s": pared, "gpu_pico_mb": gpu.pico_mb, "gpu_base_mb": gpu.base_mb}


def medir(nombre, cmd, env):
    """lanzar() con reanudación: si el JSON ya existe y REPETIR es False, lo reutiliza."""
    ruta = RESULTADOS / f"{nombre}.json"
    if ruta.is_file() and not REPETIR:
        print(f"{nombre}: ya medido, se reutiliza {ruta.name}")
        return json.loads(ruta.read_text())
    ruta.unlink(missing_ok=True)
    externa = lanzar(nombre, cmd, env)
    datos = json.loads(ruta.read_text())
    datos["medicion_externa"] = externa
    ruta.write_text(json.dumps(datos, indent=2, ensure_ascii=False))
    return datos


CONFIGS = {d: config_digitalizador(d) for d in DISPOSITIVOS}
print({d: str(p) for d, p in CONFIGS.items()})

### 4a. Tiempo por etapa

Mismo perfil que `docs/RENDIMIENTO.md`, sección 1, sobre todas las imágenes: un solo proceso por dispositivo que carga los modelos una vez y digitaliza las imágenes una tras otra; después ECGFounder (camino `rhythm`) sobre cada CSV.

En GPU se interpretan también los CSV que salieron de la CPU, para separar las dos fuentes posibles de diferencia en la sección 5: la digitalización y ECGFounder.

La primera imagen de cada dispositivo incluye calentamiento (en GPU, CUDA elige sus rutinas la primera vez), por eso la tabla da también la media sin ella.

In [ ]:
ETAPAS = {}
for dispositivo in DISPOSITIVOS:
    nombre = f"etapas_{dispositivo}"
    cmd = [
        str(VENV_PY), str(MEDIR / "medir_etapas.py"),
        "--digitalizador", str(DIGITALIZADOR),
        "--config", str(CONFIGS[dispositivo]),
        "--imagenes", str(IMAGENES),
        "--salida", str(RESULTADOS / nombre),
        "--dispositivo", dispositivo,
        "--hilos", str(HILOS_CPU),
        "--json", str(RESULTADOS / f"{nombre}.json"),
        "--pesos", str(PESOS_1LEAD),
        "--png-muestras", str(PNG_MUESTRAS),
    ]  # fmt: skip
    if dispositivo != "cpu":
        cmd += ["--interpretar-extra", str(RESULTADOS / "etapas_cpu")]
    try:
        ETAPAS[dispositivo] = medir(nombre, cmd, entorno_medicion(dispositivo))
    except Exception as exc:
        if dispositivo == "cpu":
            raise
        # Un fallo en GPU no tira lo medido en CPU: se anota y el resto del notebook sigue sin GPU.
        FALLOS[nombre] = str(exc)
        print(f"FALLO en {nombre}; se sigue solo con CPU.\n{exc}")

In [ ]:
ARRANQUE = ["Import de torch", "Inicio de CUDA", "Import del digitalizador", "Carga de los dos U-Net"]
POR_IMAGEN = [
    "Decodificar imagen",
    "Remuestreo inicial",
    "Segmentación (U-Net)",
    "Detección de perspectiva",
    "Recorte",
    "Remuestreo de mapas",
    "Búsqueda de tamaño de píxel",
    "Corrección de deformación (desactivada)",
    "Extracción de señal",
    "Identificación de formato (segundo U-Net)",
    "Guardar CSV",
    "Guardar metadatos",
    "Total por imagen (sin PNG)",
    "Guardar PNG de depuración (aparte)",
]


def valores_etapa(datos, etapa, sin_primera=False):
    imagenes = [im for im in datos["imagenes"] if not (sin_primera and im["orden"] == 0)]
    if etapa == "Total por imagen (sin PNG)":
        return [im["total_s"] for im in imagenes]
    if etapa == "Guardar PNG de depuración (aparte)":
        return [im["png_depuracion_s"] for im in imagenes]
    if etapa == "ECGFounder: carga":
        return [datos["ecgfounder"].get("carga_s")]
    if etapa == "ECGFounder: inferencia por registro":
        return list((datos["ecgfounder"].get("inferencia_s_propias") or {}).values())
    return [im["etapas"].get(etapa) for im in imagenes]


def tabla_etapas(etapas):
    filas = []
    for etapa in ARRANQUE + POR_IMAGEN + ["ECGFounder: carga", "ECGFounder: inferencia por registro"]:
        una_vez = etapa in ARRANQUE or etapa == "ECGFounder: carga"
        fila = {"Etapa": etapa, "Cuándo": "una vez por proceso" if una_vez else "por imagen, media"}
        for dispositivo, datos in etapas.items():
            if etapa in ARRANQUE:
                fila[f"{NOMBRE[dispositivo]} (s)"] = datos["proceso"].get(etapa)
                fila[f"{NOMBRE[dispositivo]} sin la primera imagen (s)"] = None
            else:
                fila[f"{NOMBRE[dispositivo]} (s)"] = media(valores_etapa(datos, etapa))
                fila[f"{NOMBRE[dispositivo]} sin la primera imagen (s)"] = media(valores_etapa(datos, etapa, True))
        filas.append(fila)
    return pd.DataFrame(filas)


TABLA_ETAPAS = tabla_etapas(ETAPAS)
display(TABLA_ETAPAS.round(3))

def total_por_imagen(dispositivo):
    """Media sin la primera imagen, o con ella si solo hay una."""
    datos = ETAPAS[dispositivo]
    return media(valores_etapa(datos, "Total por imagen (sin PNG)", True)) or media(
        valores_etapa(datos, "Total por imagen (sin PNG)")
    )


if "cuda" in ETAPAS:
    cpu, gpu = total_por_imagen("cpu"), total_por_imagen("cuda")
    print(f"Digitalización por imagen: CPU {cpu:.1f} s, GPU {gpu:.1f} s, {cpu / gpu:.1f} veces más rápida en GPU.")
png = media(valores_etapa(ETAPAS["cpu"], "Guardar PNG de depuración (aparte)"))
if png is not None:
    print(f"La imagen de depuración costaría {png:.1f} s más por estudio en CPU; no está incluida en los totales.")

### 4b. Tiempo total por estudio, en frío y en caliente

Ahora como en producción: cada estudio es una llamada al pipeline, igual que las hace el worker de api-EKG, y se mide lo que espera el estudiante, de la imagen al resultado de ECGFounder. Un proceso nuevo por cada combinación de dispositivo y modo, para que el primer estudio pague el arranque de verdad.

Los dos modos del digitalizador (`ECG_DIGITIZER_MODE`):

- `subprocess`: arranca un proceso del digitalizador por estudio, así que cada estudio vuelve a cargar sus modelos.
- `persistent` (el de producción): arranca el digitalizador una vez y lo deja con los modelos cargados. Solo el primer estudio paga la carga.

Cómo leer la tabla:

- Primer estudio desde que arranca el proceso: lo que esperaría el primer estudiante con un servidor recién encendido, o con un contenedor nuevo. No incluye lo que tarda la plataforma en levantar el contenedor.
- Estudios siguientes: lo que se espera por estudio con el servidor ya caliente.
- Sobrecoste del arranque en frío: el primer estudio menos la repetición de la misma imagen al final.

In [ ]:
MODOS = ["subprocess", "persistent"]
ESTUDIOS = {}
for dispositivo in [d for d in DISPOSITIVOS if d in ETAPAS]:
    for modo in MODOS:
        nombre = f"estudios_{dispositivo}_{modo}"
        cmd = [
            str(VENV_PY), str(MEDIR / "medir_estudios.py"),
            "--imagenes", *[str(IMAGENES / r) for r in SECUENCIA_ESTUDIOS],
            "--salida", str(RESULTADOS / nombre),
            "--config", str(CONFIGS[dispositivo]),
            "--dispositivo", dispositivo,
            "--pesos", str(PESOS_1LEAD),
            "--json", str(RESULTADOS / f"{nombre}.json"),
        ]  # fmt: skip
        try:
            ESTUDIOS[(dispositivo, modo)] = medir(nombre, cmd, entorno_medicion(dispositivo, modo))
        except Exception as exc:
            FALLOS[nombre] = str(exc)
            print(f"FALLO en {nombre}.\n{exc}")

In [ ]:
def resumen_estudios(dispositivo, modo, datos):
    estudios = datos["estudios"]
    primero, ultimo = estudios[0], estudios[-1]
    repetida = len(estudios) > 1 and primero["imagen"] == ultimo["imagen"]
    frio = primero["fin_desde_inicio_proceso_s"]
    return {
        "Dispositivo": NOMBRE[dispositivo],
        "Modo": modo,
        "Primer estudio desde que arranca el proceso (s)": frio,
        "Estudios siguientes, media (s)": media([e["total_s"] for e in estudios[1:]]),
        "Repetición de la primera imagen (s)": ultimo["total_s"] if repetida else None,
        "Sobrecoste del arranque en frío (s)": frio - ultimo["total_s"] if repetida else None,
        "Carga de ECGFounder (s)": primero["carga_ecgfounder_s"],
        "Carga del digitalizador persistente (s)": datos.get("carga_digitalizador_persistente_s"),
        "Estudios": len(estudios),
        "Resultado": "; ".join(f"{e['formato']} / {e['top1']}" + (f" ERROR {e['error']}" if e["error"] else "") for e in estudios),
    }


TABLA_ESTUDIOS = pd.DataFrame([resumen_estudios(d, m, datos) for (d, m), datos in ESTUDIOS.items()])
display(TABLA_ESTUDIOS.round(2))

detalle = []
for (dispositivo, modo), datos in ESTUDIOS.items():
    for e in datos["estudios"]:
        detalle.append(
            {
                "Dispositivo": NOMBRE[dispositivo],
                "Modo": modo,
                "Orden": e["orden"],
                "Imagen": "/".join(Path(e["imagen"]).parts[-2:]),
                "Digitalización (s)": e["digitalizacion_s"],
                "ECGFounder (s)": e["interpretacion_s"],
                "Total (s)": e["total_s"],
            }
        )
display(pd.DataFrame(detalle).round(2))

### 4c. Memoria pico

- RSS pico: la memoria RAM máxima que ocupó cada proceso. En los estudios hay dos procesos, el del pipeline con ECGFounder y el del digitalizador, y la máquina necesita la suma.
- GPU según PyTorch: la memoria que el digitalizador pidió para sus tensores (asignada) y la que tenía reservada.
- GPU según nvidia-smi: lo que ocupaba la tarjeta entera, contextos de CUDA incluidos, medido cada medio segundo. Es la cifra a comparar con la memoria de la GPU de producción.

In [ ]:
def resta(a, b):
    return a - b if a is not None and b is not None else None


filas = []
for dispositivo, datos in ETAPAS.items():
    externa = datos.get("medicion_externa") or {}
    filas.append(
        {
            "Medición": f"Perfil por etapas, {NOMBRE[dispositivo]}",
            "RSS pico del proceso (MB)": datos["proceso"].get("RSS pico MB"),
            "RSS pico del digitalizador aparte (MB)": None,
            "GPU asignada por PyTorch (MB)": datos["proceso"].get("GPU memoria pico asignada MB"),
            "GPU reservada por PyTorch (MB)": datos["proceso"].get("GPU memoria pico reservada MB"),
            "GPU pico nvidia-smi sobre la base (MB)": resta(externa.get("gpu_pico_mb"), externa.get("gpu_base_mb")),
        }
    )
for (dispositivo, modo), datos in ESTUDIOS.items():
    externa = datos.get("medicion_externa") or {}
    filas.append(
        {
            "Medición": f"Estudios, {NOMBRE[dispositivo]}, {modo}",
            "RSS pico del proceso (MB)": datos.get("rss_pico_proceso_mb"),
            "RSS pico del digitalizador aparte (MB)": datos.get("rss_pico_digitalizador_mb"),
            "GPU asignada por PyTorch (MB)": datos.get("gpu_memoria_pico_ecgfounder_mb"),
            "GPU reservada por PyTorch (MB)": None,
            "GPU pico nvidia-smi sobre la base (MB)": resta(externa.get("gpu_pico_mb"), externa.get("gpu_base_mb")),
        }
    )
TABLA_MEMORIA = pd.DataFrame(filas)
display(TABLA_MEMORIA.round(0))
print("En los estudios, 'GPU asignada por PyTorch' es solo la de ECGFounder; la del digitalizador está en el perfil por etapas.")

## 5. Calidad: CPU frente a GPU

Una GPU hace las mismas cuentas en otro orden y con otras rutinas, así que los números pueden cambiar en los últimos decimales. Lo que importa es si eso cambia la señal digitalizada o lo que dice ECGFounder.

Qué se compara, sobre las mismas imágenes:

- Contra la señal original de PTB-XL, como en el estudio de escalas: correlación r y SNR por derivación, tras alinear hasta 50 ms. El SNR principal se calcula con la media de cada traza restada, para no castigar un desplazamiento constante de la línea base que no cambia la forma. Por registro, la mediana sobre las derivaciones; por variante, la media de esas medianas.
- El formato detectado, si la tira de ritmo quedó en II y las compuertas de calidad.
- La diferencia máxima absoluta entre los CSV de CPU y de GPU, en microvoltios, y cuántas muestras tienen hueco en uno y no en el otro.
- ECGFounder: si coinciden las 10 primeras etiquetas, las 3 primeras y los hallazgos marcados por los umbrales, y la mayor diferencia de puntuación entre las 150 clases. Se mira de dos formas: de punta a punta (cada dispositivo con sus propios CSV) y con el mismo CSV de CPU interpretado en los dos dispositivos.

Tolerancia, la misma del estudio de escalas (`docs/RENDIMIENTO.md`, sección 2). La GPU se acepta si, frente a la CPU:

- el SNR medio no empeora más de 0,5 dB en ninguna variante,
- la r media no empeora más de 0,002 en ninguna variante,
- no aparece ninguna compuerta nueva, ningún formato distinto y ninguna tira fuera de II.

0,5 dB y 0,002 son del orden del ruido de la propia medición. Las compuertas no tienen margen: una tira atribuida a otra derivación cambia el trazo que lee el modelo. Las observaciones de ECGFounder se informan aparte y no entran en la tolerancia, que es la del estudio de escalas.

In [ ]:
(RESULTADOS / "registros.json").write_text(
    json.dumps({i: {"grupo": g, "filename_hr": ruta_wfdb(i)} for i, g in REGISTROS.items()}, indent=2)
)
cmd = [
    str(VENV_PY), str(MEDIR / "calidad.py"),
    "--registros", str(RESULTADOS / "registros.json"),
    "--etapas-cpu", str(RESULTADOS / "etapas_cpu.json"),
    "--wfdb-root", str(PTBXL),
    "--json", str(RESULTADOS / "calidad.json"),
]  # fmt: skip
if "cuda" in ETAPAS:
    cmd += ["--etapas-gpu", str(RESULTADOS / "etapas_cuda.json")]
if ESTUDIOS:
    cmd += ["--estudios", *[str(RESULTADOS / f"estudios_{d}_{m}.json") for d, m in ESTUDIOS]]
r = subprocess.run(cmd, capture_output=True, text=True, cwd=MEDIR)
(RESULTADOS / "logs" / "calidad.log").write_text(r.stdout + r.stderr)
if r.returncode != 0:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise RuntimeError("calidad.py falló")
CALIDAD = json.loads((RESULTADOS / "calidad.json").read_text())
print("Calidad calculada.")

In [ ]:
ETIQUETAS_VARIANTE = {
    "registros": "Registros",
    "snr_cpu_db": "SNR medio CPU (dB)",
    "snr_gpu_db": "SNR medio GPU (dB)",
    "dif_snr_db": "Diferencia SNR GPU - CPU (dB)",
    "r_cpu": "r media CPU",
    "r_gpu": "r media GPU",
    "dif_r": "Diferencia r GPU - CPU",
    "tiras_en_II_cpu": "Tira en II, CPU",
    "tiras_en_II_gpu": "Tira en II, GPU",
    "degradados_cpu": "Con compuertas, CPU",
    "degradados_gpu": "Con compuertas, GPU",
    "formatos_distintos": "Formatos distintos",
    "compuertas_nuevas": "Registros con compuertas nuevas",
    "tiras_fuera_de_II_nuevas": "Tiras que dejan II",
    "csv_identicos": "CSV idénticos",
    "csv_max_abs_uv": "Diferencia máxima entre CSV (µV)",
    "obs_top3_iguales": "ECGFounder: top 3 igual",
    "obs_top10_iguales": "ECGFounder: top 10 igual",
    "obs_marcadas_iguales": "ECGFounder: marcadas iguales",
    "cumple_snr": "Cumple SNR",
    "cumple_r": "Cumple r",
    "cumple_compuertas": "Cumple compuertas, formato y tira",
}
por_variante = pd.DataFrame(CALIDAD["por_variante"])
por_variante = por_variante.loc[[k for k in ETIQUETAS_VARIANTE if k in por_variante.index]]
por_variante.index = [ETIQUETAS_VARIANTE[k] for k in por_variante.index]
display(por_variante)

COLUMNAS_REGISTRO = [
    "variante", "ecg_id", "grupo", "snr_mac_db", "r_mac", "snr_cpu_db", "snr_gpu_db", "dif_snr_db",
    "r_cpu", "r_gpu", "dif_r", "formato_cpu", "formato_gpu", "tira_en_II_cpu", "tira_en_II_gpu",
    "compuertas_cpu", "compuertas_gpu", "csv_max_abs_uv", "csv_huecos_distintos",
    "obs_top3_iguales", "obs_marcadas_iguales", "obs_max_dif_puntuacion",
    "ecgfounder_mismo_csv_marcadas_iguales", "ecgfounder_mismo_csv_max_dif_puntuacion",
]  # fmt: skip
por_registro = pd.DataFrame(CALIDAD["por_registro"])
display(por_registro[[c for c in COLUMNAS_REGISTRO if c in por_registro.columns]].round(4))
print("snr_mac_db y r_mac son la línea base del estudio de escalas en el Mac, solo como referencia.")

if CALIDAD.get("consistencia_estudios"):
    consistencia = pd.DataFrame(CALIDAD["consistencia_estudios"])
    iguales = int(consistencia["identicos"].sum())
    print(f"\nCSV de los estudios (pipeline.run) frente a los del perfil por etapas: {iguales} de {len(consistencia)} idénticos.")
    if iguales < len(consistencia):
        display(consistencia)

if CALIDAD["cumple"] is None:
    VEREDICTO_CALIDAD = "sin GPU medida"
    print("\nNo hay medición de GPU con la que comparar.")
elif CALIDAD["cumple"]:
    VEREDICTO_CALIDAD = "igual a CPU dentro de la tolerancia"
    print("\nLa GPU cumple la tolerancia: la calidad de la digitalización no cambia.")
else:
    VEREDICTO_CALIDAD = "NO cumple la tolerancia"
    print("\nLa GPU NO cumple la tolerancia:")
    for incumplimiento in CALIDAD["incumplimientos"]:
        print(" -", incumplimiento)

## 6. La clase: 10 estudios a la vez

Situación: en clase, 10 estudiantes envían su ECG en el mismo momento. ¿Cuánto espera el último en recibir su resultado? Se simula con los tiempos medidos arriba, modo persistente, en tres formas de servir el análisis:

1. Una VM con 2 workers en CPU. La máquina está siempre encendida y los dos workers ya tienen los modelos cargados, así que no hay arranque en frío. Atienden de dos en dos: el estudiante 10 espera cinco estudios seguidos.
2. Cloud Run en CPU, un contenedor por estudio. Los 10 se atienden en paralelo, pero cada contenedor nace en frío y paga el arranque.
3. Cloud Run con GPU, cobrado por segundo, un contenedor por estudio. Igual que el anterior, con los tiempos de GPU y su arranque en frío.

Supuestos, para leer la tabla con cuidado:

- Cada worker o contenedor tiene tantas vCPU como hilos se midieron aquí (`HILOS_CPU`). La VM necesitaría `WORKERS_VM` veces esas vCPU y esa memoria.
- Colab no es el servidor de producción: su CPU es una Xeon compartida y su GPU una T4. Cloud Run ofrece GPU L4, que suele ser más rápida que la T4, así que el tiempo de GPU de aquí es conservador.
- No incluye lo que tarda la plataforma en levantar el contenedor ni en descargar la imagen; si conoces esa cifra, ponla en `ARRANQUE_CONTENEDOR_S` y vuelve a ejecutar desde aquí.
- Si la cuota de GPU del proyecto permite menos de 10 instancias, baja `MAX_INSTANCIAS_GPU`: los estudios que no quepan esperan a que se libere una.

In [ ]:
def simular_cola(n_estudios, servidores, t_frio, t_caliente, calientes_al_empezar):
    """Momento en que cada estudiante recibe su resultado si los n estudios llegan a la vez.

    Cada servidor (un worker o un contenedor) atiende un estudio cada vez, por orden de llegada.
    Un servidor que todavía no ha atendido ninguno tarda t_frio, salvo que ya estuviera encendido
    y con los modelos cargados; después, t_caliente.
    """
    libre_en = [0.0] * servidores
    usado = [calientes_al_empezar] * servidores
    fin = []
    for _ in range(n_estudios):
        k = min(range(servidores), key=lambda j: libre_en[j])
        libre_en[k] += t_caliente if usado[k] else t_frio
        usado[k] = True
        fin.append(libre_en[k])
    return fin


def tiempos_medidos(dispositivo, modo="persistent"):
    datos = ESTUDIOS.get((dispositivo, modo))
    if not datos:
        return None
    resumen = resumen_estudios(dispositivo, modo, datos)
    return {
        "frio": resumen["Primer estudio desde que arranca el proceso (s)"],
        "caliente": resumen["Estudios siguientes, media (s)"],
    }


ESCENARIOS = [
    (f"VM con {WORKERS_VM} workers en CPU", "cpu", WORKERS_VM, True, 0.0),
    ("Cloud Run en CPU, un contenedor por estudio", "cpu", min(N_ESTUDIANTES, MAX_INSTANCIAS_CPU), False, ARRANQUE_CONTENEDOR_S),
    ("Cloud Run con GPU, un contenedor por estudio", "cuda", min(N_ESTUDIANTES, MAX_INSTANCIAS_GPU), False, ARRANQUE_CONTENEDOR_S),
]
COLUMNA_ESPERA = f"Espera del estudiante {N_ESTUDIANTES} (s)"
FILAS_CLASE = []
for escenario, dispositivo, servidores, calientes, arranque_contenedor in ESCENARIOS:
    t = tiempos_medidos(dispositivo)
    fila = {
        "Escenario": escenario,
        "Dispositivo medido": NOMBRE.get(dispositivo, dispositivo) if t else "no medido",
        "Estudios en paralelo": servidores,
        "Estudio en caliente (s)": None,
        "Estudio en frío (s)": None,
        COLUMNA_ESPERA: None,
        "Espera media (s)": None,
        "Calidad": ("referencia" if dispositivo == "cpu" else VEREDICTO_CALIDAD) if t else "no medida",
    }
    if t:
        frio = t["frio"] + arranque_contenedor
        fin = simular_cola(N_ESTUDIANTES, servidores, frio, t["caliente"], calientes)
        fila.update(
            {
                "Estudio en caliente (s)": t["caliente"],
                "Estudio en frío (s)": None if calientes else frio,
                COLUMNA_ESPERA: fin[-1],
                "Espera media (s)": fmean(fin),
            }
        )
    FILAS_CLASE.append(fila)
TABLA_FINAL = pd.DataFrame(FILAS_CLASE)
display(TABLA_FINAL.round(1))


def conclusion():
    medidos = [f for f in FILAS_CLASE if f[COLUMNA_ESPERA] is not None]
    mejor = min(medidos, key=lambda f: f[COLUMNA_ESPERA])
    lineas = [
        f"Si {N_ESTUDIANTES} estudiantes envían su ECG a la vez, el último recibe su resultado antes con "
        f"«{mejor['Escenario']}»: unos {mejor[COLUMNA_ESPERA]:.0f} s."
    ]
    for f in medidos:
        detalle = f"{f['Estudio en caliente (s)']:.0f} s por estudio con el servidor caliente"
        if f["Estudio en frío (s)"] is not None:
            detalle += f", {f['Estudio en frío (s)']:.0f} s si el contenedor arranca en frío"
        lineas.append(f"- {f['Escenario']}: el estudiante {N_ESTUDIANTES} espera {f[COLUMNA_ESPERA]:.0f} s ({detalle}).")
    for f in FILAS_CLASE:
        if f[COLUMNA_ESPERA] is None:
            lineas.append(f"- {f['Escenario']}: no se pudo medir en esta sesión.")
    if CALIDAD["cumple"] is True:
        lineas.append("Calidad: la GPU da la misma digitalización dentro de la tolerancia del estudio de escalas.")
    elif CALIDAD["cumple"] is False:
        lineas.append("Calidad: la GPU NO cumple la tolerancia (sección 5); no conviene elegirla hasta entender por qué.")
    obs = [v.get("obs_marcadas_iguales") for v in CALIDAD["por_variante"].values() if "obs_marcadas_iguales" in v]
    if obs:
        total = sum(v["registros"] for v in CALIDAD["por_variante"].values())
        lineas.append(f"ECGFounder marca los mismos hallazgos en CPU y en GPU en {sum(obs)} de {total} imágenes.")
    rss = [v for v in TABLA_MEMORIA["RSS pico del digitalizador aparte (MB)"].dropna()]
    if rss:
        lineas.append(
            f"Memoria: el digitalizador llega a unos {max(rss) / 1024:.1f} GB de RAM, más el proceso del pipeline; "
            "cada contenedor o worker necesita al menos eso."
        )
    lineas.append(
        "Límites: Colab no es producción (CPU compartida, GPU T4 frente a la L4 de Cloud Run) y no se mide el "
        "arranque del contenedor en sí. Los tiempos sirven para comparar las tres opciones entre sí."
    )
    return "\n".join(lineas)


CONCLUSION = conclusion()
print(CONCLUSION)

## 7. Resultados para descargar

Se guardan en `/content/resultados` y se descargan juntos en `resultados_gpu_cpu.zip`:

- `mediciones.csv`: todas las mediciones en formato largo, una por fila (sección, dispositivo, modo, elemento, métrica, valor, unidad).
- `mediciones.json`: lo mismo con todo el detalle, más el entorno, los parámetros y los fallos, si los hubo.
- `tabla_final.csv` y `tabla_final.md`: la tabla de la clase y la conclusión.
- Los JSON de cada medición y sus logs, por si hay que revisar algo.

Si la descarga no arranca sola, están en el panel de archivos de la izquierda, en `/content`.

In [ ]:
import csv
import datetime
import zipfile

FILAS_MEDICIONES = []


def anotar(seccion, dispositivo, modo, elemento, metrica, valor, unidad=""):
    if isinstance(valor, (bool, int, float, str)) or valor is None:
        FILAS_MEDICIONES.append(
            {
                "seccion": seccion,
                "dispositivo": dispositivo,
                "modo": modo,
                "elemento": elemento,
                "metrica": metrica,
                "valor": valor,
                "unidad": unidad,
            }
        )


for dispositivo, datos in ETAPAS.items():
    for metrica, valor in datos["proceso"].items():
        anotar("etapas_proceso", dispositivo, "un_proceso", "", metrica, valor, "MB" if "MB" in metrica else "s")
    for imagen in datos["imagenes"]:
        anotar("etapas_imagen", dispositivo, "un_proceso", imagen["imagen"], "Total por imagen (sin PNG)", imagen["total_s"], "s")
        for etapa, valor in imagen["etapas"].items():
            anotar("etapas_imagen", dispositivo, "un_proceso", imagen["imagen"], etapa, valor, "s")
        if imagen["png_depuracion_s"] is not None:
            anotar("etapas_imagen", dispositivo, "un_proceso", imagen["imagen"], "Guardar PNG de depuración (aparte)", imagen["png_depuracion_s"], "s")
    anotar("ecgfounder", dispositivo, "un_proceso", "", "carga", datos["ecgfounder"].get("carga_s"), "s")
    for clave in ("propias", "extra"):
        for registro, valor in (datos["ecgfounder"].get(f"inferencia_s_{clave}") or {}).items():
            anotar("ecgfounder", dispositivo, "un_proceso", registro, f"inferencia ({clave})", valor, "s")
    externa = datos.get("medicion_externa") or {}
    anotar("memoria", dispositivo, "un_proceso", "", "GPU pico nvidia-smi", externa.get("gpu_pico_mb"), "MB")
    anotar("memoria", dispositivo, "un_proceso", "", "GPU base nvidia-smi", externa.get("gpu_base_mb"), "MB")

for (dispositivo, modo), datos in ESTUDIOS.items():
    for e in datos["estudios"]:
        elemento = f"{e['orden']}:" + "/".join(Path(e["imagen"]).parts[-2:])
        for metrica in ("digitalizacion_s", "interpretacion_s", "carga_ecgfounder_s", "total_s", "fin_desde_inicio_proceso_s"):
            anotar("estudios", dispositivo, modo, elemento, metrica, e[metrica], "s")
    for metrica, valor in resumen_estudios(dispositivo, modo, datos).items():
        anotar("estudios_resumen", dispositivo, modo, "", metrica, valor, "s" if "(s)" in metrica else "")
    for metrica in ("rss_pico_proceso_mb", "rss_pico_digitalizador_mb", "gpu_memoria_pico_ecgfounder_mb"):
        anotar("memoria", dispositivo, modo, "", metrica, datos.get(metrica), "MB")
    externa = datos.get("medicion_externa") or {}
    anotar("memoria", dispositivo, modo, "", "GPU pico nvidia-smi", externa.get("gpu_pico_mb"), "MB")

for fila in CALIDAD["por_registro"]:
    elemento = f"{fila['variante']}/{fila['ecg_id']}"
    for metrica, valor in fila.items():
        if metrica not in ("variante", "ecg_id"):
            anotar("calidad_registro", "", "", elemento, metrica, valor)
for variante, valores in CALIDAD["por_variante"].items():
    for metrica, valor in valores.items():
        anotar("calidad_variante", "", "", variante, metrica, valor)
anotar("calidad", "", "", "", "cumple_tolerancia", CALIDAD["cumple"])

for fila in FILAS_CLASE:
    for metrica, valor in fila.items():
        if metrica != "Escenario":
            anotar("clase", "", "", fila["Escenario"], metrica, valor, "s" if "(s)" in metrica else "")

with open(RESULTADOS / "mediciones.csv", "w", newline="") as fh:
    escritor = csv.DictWriter(fh, fieldnames=list(FILAS_MEDICIONES[0]))
    escritor.writeheader()
    escritor.writerows(FILAS_MEDICIONES)

MEDICIONES = {
    "generado": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "entorno": ENTORNO,
    "parametros": {
        "rama_pipeline": RAMA_PIPELINE,
        "commit_digitalizador": COMMIT_DIGITALIZADOR,
        "commit_image_kit": COMMIT_IMAGE_KIT,
        "modo_rapido": MODO_RAPIDO,
        "hilos_cpu": HILOS_CPU,
        "registros": REGISTROS,
        "secuencia_estudios": SECUENCIA_ESTUDIOS,
        "n_estudiantes": N_ESTUDIANTES,
        "workers_vm": WORKERS_VM,
        "max_instancias_cpu": MAX_INSTANCIAS_CPU,
        "max_instancias_gpu": MAX_INSTANCIAS_GPU,
        "arranque_contenedor_s": ARRANQUE_CONTENEDOR_S,
    },
    "fallos": FALLOS,
    "etapas": ETAPAS,
    "estudios": {f"{d}_{m}": datos for (d, m), datos in ESTUDIOS.items()},
    "calidad": CALIDAD,
    "clase": {"tabla": FILAS_CLASE, "conclusion": CONCLUSION},
}
(RESULTADOS / "mediciones.json").write_text(json.dumps(MEDICIONES, indent=2, ensure_ascii=False, default=str))

TABLA_FINAL.to_csv(RESULTADOS / "tabla_final.csv", index=False)


def celda_md(valor):
    if valor is None or (isinstance(valor, float) and valor != valor):
        return ""
    return f"{valor:.1f}" if isinstance(valor, float) else str(valor)


columnas = list(TABLA_FINAL.columns)
lineas_md = ["| " + " | ".join(columnas) + " |", "|" + "---|" * len(columnas)]
for fila in FILAS_CLASE:
    lineas_md.append("| " + " | ".join(celda_md(fila[c]) for c in columnas) + " |")
(RESULTADOS / "tabla_final.md").write_text("\n".join(lineas_md) + "\n\n" + CONCLUSION + "\n")

ZIP = CONTENT / "resultados_gpu_cpu.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for ruta in sorted(RESULTADOS.glob("*.json")) + sorted(RESULTADOS.glob("*.csv")) + sorted(RESULTADOS.glob("*.md")):
        zf.write(ruta, ruta.name)
    for ruta in sorted((RESULTADOS / "logs").glob("*.log")):
        zf.write(ruta, f"logs/{ruta.name}")
print(f"{len(FILAS_MEDICIONES)} mediciones guardadas en {RESULTADOS}; zip: {ZIP} ({ZIP.stat().st_size / 1024:.0f} KB)")
if FALLOS:
    print("Fallos durante la sesión:", json.dumps(FALLOS, indent=2, ensure_ascii=False))

try:
    from google.colab import files

    files.download(str(ZIP))
except ImportError:
    print("Fuera de Colab: descarga el zip a mano.")